# Student Payment Data Cleaning and Reconciliation

**Tools:** Python, Pandas, Jupyter Notebook, Excel

This notebook documents the original exploratory cleaning workflow across the workbook's sheets, including data-quality checks, record matching, payment reconciliation, and export steps. Some cells represent iterative investigation and manual corrections from the original working notebook; review and run sequentially against an authorised local copy of the source workbook before relying on the outputs.

## Before running
1. Place the source workbook in `data/raw/` (it is intentionally not committed to this public repository).
2. Update the workbook filename in the first code cell if needed.
3. Do not publish student names, phone numbers, IDs, or payment-level data.


In [ ]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("data")
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

file_path = RAW_DIR / "inetz worksheet new.xlsx"
print(f"Source workbook configured: {file_path}")

## 1. Load the workbook and inspect source sheets

Workbook loading, dimensions, and initial inspection.

In [ ]:
import pandas as pd

file_path = RAW_DIR / "inetz worksheet new.xlsx"

internship = pd.read_excel(file_path, sheet_name="Internship")
registered_students = pd.read_excel(file_path, sheet_name="Registered Students")
payment_entries = pd.read_excel(file_path, sheet_name="Payment Entries")
student_summary = pd.read_excel(file_path, sheet_name="Student Summary")
registered_intern_details = pd.read_excel(file_path, sheet_name="Registered Intern Details")
course = pd.read_excel(file_path, sheet_name="Course")

In [ ]:
print("Internship:", internship.shape)
print("Registered Students:", registered_students.shape)
print("Payment Entries:", payment_entries.shape)
print("Student Summary:", student_summary.shape)
print("Registered Intern Details:", registered_intern_details.shape)
print("Course:", course.shape)

In [ ]:
internship.head()

In [ ]:
internship.info()

In [ ]:
internship.columns.tolist()

In [ ]:
internship.isnull().sum()

In [ ]:
internship.duplicated().sum()

In [ ]:
internship = internship.dropna(how="all").reset_index(drop=True)

In [ ]:
internship.shape

In [ ]:
internship.head(10)

## 2. Clean the Internship worksheet

Header repair, column naming, data types, text cleanup, and quality checks.

In [ ]:
internship = pd.read_excel(file_path, sheet_name="Internship")

In [ ]:
internship = pd.read_excel(
    file_path,
    sheet_name="Internship",
    header=[0, 1]
)

In [ ]:
internship.head()

In [ ]:
internship.columns

In [ ]:
internship = pd.read_excel(
    file_path,
    sheet_name="Internship",
    header=[2, 3]
)

In [ ]:
internship.columns

In [ ]:
new_columns = []

for main, sub in internship.columns:
    main = str(main).strip()
    sub = str(sub).strip()

    if sub.startswith("Unnamed") or sub == "nan":
        new_columns.append(main)
    else:
        new_columns.append(f"{main} {sub}")

internship.columns = new_columns

In [ ]:
print(internship.columns.tolist())

In [ ]:
internship.head()

In [ ]:
for i, col in enumerate(internship.columns):
    print(i, col)

In [ ]:
internship.columns = [
    col.split(" ", 1)[1] if " " in col else col
    for col in internship.columns
]

In [ ]:
internship.columns = [
    'S.No',
    'DOJ',
    'Group No',
    'Name',
    'Phone number',
    'College',
    'Duration',
    'Total billing',
    'Installment 1 Date',
    'Installment 1 Bill No',
    'Installment 1 GPay',
    'Installment 1 Cash',
    'Installment 2 Date',
    'Installment 2 Bill No',
    'Installment 2 GPay',
    'Installment 2 Cash',
    'Installment 3 Date',
    'Installment 3 Bill No',
    'Installment 3 GPay',
    'Installment 3 Cash',
    'Total collection',
    'Pending amount',
    'Fees Status',
    'Certificate Status 1',
    'Certificate Status 2',
    'Certificate Status 3',
    'Certificate Status 4',
    'Certificate Status 5',
    'Certificate Status 6',
    'Certificate Status 7',
    'Certificate Status 8',
    'Certificate Status 9',
    'Certificate Status 10',
    'Certificate Status 11',
    'Certificate Status 12',
    'Certificate Status 13',
    'Certificate Status 14',
    'Certificate Status 15',
    'Certificate Status 16',
    'Certificate Status 17',
    'Certificate Status 18',
    'Certificate Status 19',
    'Certificate Status 20',
    'Certificate Status 21',
    'Certificate Status 22'
]

In [ ]:
internship.head()

In [ ]:
internship.columns = [
    'S.No',
    'DOJ',
    'Group No',
    'Name',
    'Phone Number',
    'College',
    'Duration',
    'Total Billing',
    'Installment 1 Date',
    'Installment 1 Bill No',
    'Installment 1 GPay',
    'Installment 1 Cash',
    'Installment 2 Date',
    'Installment 2 Bill No',
    'Installment 2 GPay',
    'Installment 2 Cash',
    'Installment 3 Date',
    'Installment 3 Bill No',
    'Installment 3 GPay',
    'Installment 3 Cash',
    'Total Collection',
    'Pending Amount',
    'Fees Status',
    'Certificate Status'
]

In [ ]:
internship = internship.iloc[:, :24]

In [ ]:
internship.head()

In [ ]:
internship.shape

In [ ]:
internship = internship.rename(columns={
    'Certificate Status 1': 'Certificate Status'
})

In [ ]:
internship = internship.iloc[1:].reset_index(drop=True)

In [ ]:
internship.head()

In [ ]:
internship.info()

In [ ]:
internship = internship.dropna(subset=['S.No']).reset_index(drop=True)

In [ ]:
internship.shape

In [ ]:
internship.info()

In [ ]:
internship[['Total billing', 'Total collection', 'Pending amount']].head(20)

In [ ]:
internship['Total collection'] = pd.to_numeric(
    internship['Total collection'], errors='coerce'
)

internship['Pending amount'] = pd.to_numeric(
    internship['Pending amount'], errors='coerce'
)

In [ ]:
internship[['Total billing', 'Total collection', 'Pending amount']].dtypes

In [ ]:
internship['Phone number'].head(20)

In [ ]:
internship['Phone number'].dtype

In [ ]:
internship['Phone number'] = internship['Phone number'].astype(str).str.strip()

In [ ]:
internship['Phone number'] = internship['Phone number'].apply(
    lambda x: str(x).strip() if pd.notna(x) else pd.NA
)

In [ ]:
internship['Phone number'].head(20)

In [ ]:
internship['Name'].head(20)

In [ ]:
internship['Name'].isnull().sum()

In [ ]:
internship['Name'] = internship['Name'].str.strip()

In [ ]:
internship['Group No'].value_counts(dropna=False).head(20)

In [ ]:
internship['Group No'] = internship['Group No'].str.strip()

In [ ]:
internship[internship['College'].isna()][['S.No', 'Name', 'College']]

In [ ]:
internship[internship['Duration'].isna()][['S.No', 'Name', 'Duration']]

In [ ]:
internship[['Installment 1 Date',
            'Installment 2 Date',
            'Installment 3 Date']].head(20)

In [ ]:
date_columns = [
    'Installment 1 Date',
    'Installment 2 Date',
    'Installment 3 Date'
]

for col in date_columns:
    internship[col] = pd.to_datetime(
        internship[col],
        errors='coerce',
        dayfirst=True
    )

In [ ]:
internship[date_columns].dtypes

In [ ]:
bill_columns = [
    'Installment 1 Bill No',
    'Installment 2 Bill No',
    'Installment 3 Bill No'
]

internship[bill_columns].head(20)

In [ ]:
for col in bill_columns:
    internship[col] = internship[col].apply(
        lambda x: str(x).strip() if pd.notna(x) else pd.NA
    )

In [ ]:
internship[bill_columns].dtypes

In [ ]:
payment_columns = [
    'Installment 1 GPay',
    'Installment 1 Cash',
    'Installment 2 GPay',
    'Installment 2 Cash',
    'Installment 3 GPay',
    'Installment 3 Cash'
]

internship[payment_columns].head(20)

In [ ]:
for col in payment_columns:
    internship[col] = pd.to_numeric(
        internship[col],
        errors='coerce'
    )

In [ ]:
internship[payment_columns].dtypes

In [ ]:
status_columns = [
    'Fees Status',
    'Certificate Status'
]

internship[status_columns].value_counts(dropna=False)

In [ ]:
internship['Fees Status'] = internship['Fees Status'].replace({
    'paid': 'Paid'
})

internship['Certificate Status'] = internship['Certificate Status'].replace({
    'issued': 'Issued',
    'ISSUED': 'Issued',
    'Issues': 'Issued'
})

In [ ]:
internship[['Fees Status', 'Certificate Status']].value_counts(dropna=False)

In [ ]:
internship.duplicated().sum()

In [ ]:
internship['S.No'].duplicated().sum()

## 3. Clean the Registered Students worksheet

Header and column standardisation, data types, missing values, and duplicate checks.

In [ ]:
registered_students = pd.read_excel(
    file_path,
    sheet_name="Registered Students",header = 3
)

In [ ]:
registered_students.head()

In [ ]:
registered_students.head()

In [ ]:
registered_students.info()

In [ ]:
registered_students.columns.tolist()

In [ ]:
registered_students = registered_students.dropna(
    how='all'
).reset_index(drop=True)

In [ ]:
registered_students.shape

In [ ]:
registered_students = registered_students.dropna(
    axis=1,
    how='all'
)

In [ ]:
registered_students.shape

In [ ]:
registered_students.columns.tolist()

In [ ]:
registered_students.head(70)

In [ ]:
registered_students.columns = [
    'S.No',
    'DOJ',
    'Student ID',
    'Student Name',
    'Phone Number',
    'College',
    'Duration',
    'Program',
    'Total Billing',
    'Installment 1 Date',
    'Installment 1 Bill No',
    'Installment 1 GPay',
    'Installment 1 Cash',
    'Installment 2 Date',
    'Installment 2 Bill No',
    'Installment 2 GPay',
    'Installment 2 Cash',
    'Installment 3 Date',
    'Installment 3 Bill No',
    'Installment 3 GPay',
    'Installment 3 Cash',
    'Total Collection',
    'Pending Amount',
    'Fees Status',
    'Certificate Status'
]

In [ ]:
registered_students = pd.read_excel(
    file_path,
    sheet_name="Registered Students",
    header=3
)

In [ ]:
registered_students.shape

In [ ]:
registered_students = registered_students.iloc[:, :25]

In [ ]:
registered_students.columns = [
    'S.No',
    'DOJ',
    'Student ID',
    'Student Name',
    'Phone Number',
    'College',
    'Duration',
    'Program',
    'Total Billing',
    'Installment 1 Date',
    'Installment 1 Bill No',
    'Installment 1 GPay',
    'Installment 1 Cash',
    'Installment 2 Date',
    'Installment 2 Bill No',
    'Installment 2 GPay',
    'Installment 2 Cash',
    'Installment 3 Date',
    'Installment 3 Bill No',
    'Installment 3 GPay',
    'Installment 3 Cash',
    'Total Collection',
    'Pending Amount',
    'Fees Status',
    'Certificate Status'
]

In [ ]:
registered_students.head()

In [ ]:
registered_students = registered_students.iloc[1:].reset_index(drop=True)

In [ ]:
registered_students.head()

In [ ]:
registered_students = registered_students.dropna(
    how='all'
).reset_index(drop=True)

In [ ]:
registered_students.shape

In [ ]:
registered_students.info()

In [ ]:
registered_students = registered_students.dropna(
    axis=1,
    how='all'
)

In [ ]:
registered_students.shape

In [ ]:
registered_students.columns

In [ ]:
registered_students = pd.read_excel(
    file_path,
    sheet_name="Registered Students",
    header=3
)

registered_students = registered_students.iloc[:, :25]

registered_students.columns = [
    'S.No',
    'DOJ',
    'Student ID',
    'Student Name',
    'Phone Number',
    'College',
    'Duration',
    'Program',
    'Total Billing',
    'Installment 1 Date',
    'Installment 1 Bill No',
    'Installment 1 GPay',
    'Installment 1 Cash',
    'Installment 2 Date',
    'Installment 2 Bill No',
    'Installment 2 GPay',
    'Installment 2 Cash',
    'Installment 3 Date',
    'Installment 3 Bill No',
    'Installment 3 GPay',
    'Installment 3 Cash',
    'Total Collection',
    'Pending Amount',
    'Fees Status',
    'Certificate Status'
]

registered_students = registered_students.iloc[1:].reset_index(drop=True)

In [ ]:
registered_students.columns.tolist()

In [ ]:
registered_students.dtypes

In [ ]:
registered_students.info()

In [ ]:
pd.DataFrame({
    'Data Type': registered_students.dtypes,
    'Missing Values': registered_students.isnull().sum()
})

In [ ]:
registered_students = registered_students.dropna(
    subset=['S.No']
).reset_index(drop=True)

In [ ]:
registered_students.shape

In [ ]:
pd.DataFrame({
    'Data Type': registered_students.dtypes,
    'Missing Values': registered_students.isnull().sum()
})

In [ ]:
# Integer / identifier
registered_students['S.No'] = registered_students['S.No'].astype('Int64')

# Date
registered_students['DOJ'] = pd.to_datetime(
    registered_students['DOJ'],
    errors='coerce'
)

# Text columns
text_columns = [
    'Student ID',
    'Student Name',
    'Phone Number',
    'College',
    'Duration',
    'Program',
    'Installment 1 Bill No',
    'Installment 2 Bill No',
    'Installment 3 Bill No',
    'Fees Status',
    'Certificate Status'
]

for col in text_columns:
    registered_students[col] = registered_students[col].astype('string')

# Installment dates
date_columns = [
    'Installment 1 Date',
    'Installment 2 Date',
    'Installment 3 Date'
]

for col in date_columns:
    registered_students[col] = pd.to_datetime(
        registered_students[col],
        errors='coerce',
        dayfirst=True
    )

# Monetary / payment columns
numeric_columns = [
    'Total Billing',
    'Installment 1 GPay',
    'Installment 1 Cash',
    'Installment 2 GPay',
    'Installment 2 Cash',
    'Installment 3 GPay',
    'Installment 3 Cash',
    'Total Collection',
    'Pending Amount'
]

for col in numeric_columns:
    registered_students[col] = pd.to_numeric(
        registered_students[col],
        errors='coerce'
    )

In [ ]:
registered_students.dtypes

In [ ]:
registered_students[registered_students['Phone Number'].isna()][
    ['S.No', 'Student ID', 'Student Name', 'Phone Number']
]

In [ ]:
text_columns = [
    'Student ID',
    'Student Name',
    'Phone Number',
    'College',
    'Duration',
    'Program'
]

for col in text_columns:
    registered_students[col] = registered_students[col].str.strip()

In [ ]:
registered_students.duplicated().sum()

In [ ]:
registered_students['Student ID'].duplicated().sum()

## 4. Clean the Payment Entries worksheet

Payment-entry structure, text cleanup, and duplicate Student ID review.

In [ ]:
payment_entries = pd.read_excel(
    file_path,
    sheet_name="Payment Entries",header=3
)

In [ ]:
payment_entries.head()

In [ ]:
payment_entries.head()

In [ ]:
payment_entries.info()

In [ ]:
payment_entries.columns.tolist()

In [ ]:
payment_entries['Bill No'].value_counts(dropna=False)

In [ ]:
payment_entries['Payment Mode'].value_counts(dropna=False)

In [ ]:
payment_entries['Payment Mode'] = (
    payment_entries['Payment Mode']
    .astype('string')
    .str.strip()
)

In [ ]:
payment_entries['Payment Mode'].value_counts(dropna=False)

In [ ]:
payment_entries.dtypes

In [ ]:
payment_entries.duplicated().sum()

In [ ]:
payment_entries['Student ID'].duplicated().sum()

In [ ]:
payment_entries[
    payment_entries['Student ID'].duplicated(keep=False)
].sort_values('Student ID')

In [ ]:
payment_entries.duplicated().sum()

## 5. Clean the Student Summary worksheet

Student summary inspection and consistency checks.

In [ ]:
student_summary = pd.read_excel(
    file_path,
    sheet_name="Student Summary",
    header=3
)

In [ ]:
student_summary.head()

In [ ]:
student_summary.info()

In [ ]:
student_summary.columns.tolist()

In [ ]:
student_summary[['S.No', 'Student Id', 'Student Name']].head(10)

In [ ]:
student_summary = student_summary.drop(columns=['S.No'])

In [ ]:
student_summary = student_summary.rename(columns={
    'Student Id': 'Student ID'
})

In [ ]:
student_summary.columns.tolist()

In [ ]:
student_summary.isnull().sum()

In [ ]:
student_summary.duplicated().sum()

In [ ]:
student_summary['Student ID'].duplicated().sum()

## 6. Clean Registered Intern Details

Intern details, dates, text fields, and duplicate checks.

In [ ]:
registered_intern_details = pd.read_excel(
    file_path,
    sheet_name="Registered Intern Details",
    header=0
)

In [ ]:
registered_intern_details.head()

In [ ]:
registered_intern_details.head()

In [ ]:
registered_intern_details.info()

In [ ]:
registered_intern_details.columns.tolist()

In [ ]:
registered_intern_details.columns = registered_intern_details.columns.str.strip()

In [ ]:
registered_intern_details = registered_intern_details.rename(columns={
    'NAME': 'Name',
    'COLLEGE': 'College',
    'DURATION': 'Duration',
    'CERTIFICATE ISSUED': 'Certificate Status'
})

In [ ]:
registered_intern_details.columns.tolist()

In [ ]:
registered_intern_details[
    registered_intern_details['S.No'].isna()
]

In [ ]:
registered_intern_details = registered_intern_details[
    registered_intern_details['S.No'].notna() |
    registered_intern_details['Name'].notna()
].reset_index(drop=True)

In [ ]:
registered_intern_details.shape

In [ ]:
registered_intern_details[
    registered_intern_details['Name'].str.contains(
        'Kishore', case=False, na=False
    )
]

In [ ]:
pd.DataFrame({
    'Data Type': registered_intern_details.dtypes,
    'Missing Values': registered_intern_details.isnull().sum()
})

In [ ]:
registered_intern_details['Preferred Start Date'] = pd.to_datetime(
    registered_intern_details['Preferred Start Date'],
    errors='coerce',
    dayfirst=True
)

In [ ]:
text_columns = [
    'Group No',
    'Name',
    'College',
    'Duration',
    'DOMAIN',
    'Preferred Time Slot',
    'Assigned Batch Code',
    'Batch Status',
    'Certificate Status'
]

for col in text_columns:
    registered_intern_details[col] = (
        registered_intern_details[col]
        .astype('string')
        .str.strip()
    )

In [ ]:
registered_intern_details.dtypes

In [ ]:
registered_intern_details['Batch Status'].value_counts(dropna=False)

In [ ]:
registered_intern_details['Certificate Status'].value_counts(dropna=False)

In [ ]:
registered_intern_details['Batch Status'] = (
    registered_intern_details['Batch Status'].replace('-', pd.NA)
)

registered_intern_details['Certificate Status'] = (
    registered_intern_details['Certificate Status'].replace('-', pd.NA)
)

In [ ]:
registered_intern_details['Batch Status'].value_counts(dropna=False)

In [ ]:
registered_intern_details['Certificate Status'].value_counts(dropna=False)

In [ ]:
registered_intern_details.duplicated().sum()

In [ ]:
registered_intern_details['S.No'].duplicated().sum()

In [ ]:
payment_entries = pd.read_excel(
    file_path,
    sheet_name="Payment Entries",
    header=3
)

In [ ]:
payment_entries['Payment Mode'] = (
    payment_entries['Payment Mode']
    .astype('string')
    .str.strip()
)

In [ ]:
payment_entries.shape

In [ ]:
payment_entries[
    payment_entries['Student ID'].duplicated(keep=False)
].sort_values('Student ID')

## 7. Clean the Course worksheet

Course and billing fields, dates, numeric values, and duplicate checks.

In [ ]:
course = pd.read_excel(
    file_path,
    sheet_name="Course",header = 3
)

In [ ]:
course.head()

In [ ]:
course.columns = [
    'S.No',
    'Name',
    'Program',
    'DOJ',
    'Total Billing',
    'Installment 1 Date',
    'Installment 1 Bill No',
    'Installment 1 GPay',
    'Installment 1 Cash',
    'Installment 2 Date',
    'Installment 2 Bill No',
    'Installment 2 GPay',
    'Installment 2 Cash',
    'Installment 3 Date',
    'Installment 3 Bill No',
    'Installment 3 GPay',
    'Installment 3 Cash',
    'Total Collection',
    'Pending Amount'
]

In [ ]:
course = course.iloc[1:].reset_index(drop=True)

In [ ]:
course.head()

In [ ]:
pd.DataFrame({
    'Data Type': course.dtypes,
    'Missing Values': course.isnull().sum()
})

In [ ]:
numeric_columns = [
    'Total Billing',
    'Installment 1 GPay',
    'Installment 1 Cash',
    'Installment 2 GPay',
    'Installment 2 Cash',
    'Installment 3 GPay',
    'Installment 3 Cash',
    'Total Collection',
    'Pending Amount'
]

for col in numeric_columns:
    course[col] = pd.to_numeric(
        course[col],
        errors='coerce'
    )

In [ ]:
course['DOJ'] = pd.to_datetime(
    course['DOJ'],
    errors='coerce',
    dayfirst=True
)

In [ ]:
text_columns = [
    'Name',
    'Program',
    'Installment 1 Bill No',
    'Installment 2 Bill No',
    'Installment 3 Bill No'
]

for col in text_columns:
    course[col] = (
        course[col]
        .astype('string')
        .str.strip()
    )

In [ ]:
course.dtypes

In [ ]:
for col in [
    'Installment 1 Date',
    'Installment 2 Date',
    'Installment 3 Date'
]:
    print(f'\n{col}')
    print(
        course[
            course[col].astype('string').str.contains(',', na=False)
        ][['Name', col]]
    )

In [ ]:
date_columns = [
    'Installment 1 Date',
    'Installment 2 Date',
    'Installment 3 Date'
]

for col in date_columns:
    course[col] = (
        course[col]
        .astype('string')
        .str.strip()
        .str.rstrip(',')
        .str.strip()
    )

In [ ]:
course.loc[
    course['Name'].isin([
        'Govarthini',
        'Anu',
        'Anushiya',
        'Mahalakshmi',
        'Esther Rassou'
    ]),
    ['Name', 'Installment 1 Date']
]

In [ ]:
for col in [
    'Installment 1 Bill No',
    'Installment 1 GPay',
    'Installment 1 Cash',
    'Installment 2 Bill No',
    'Installment 2 GPay',
    'Installment 2 Cash'
]:
    print(f'\n{col}')
    print(
        course[
            course[col].astype('string').str.contains(',', na=False)
        ][['Name', col]]
    )

In [ ]:
date_columns = [
    'Installment 1 Date',
    'Installment 2 Date',
    'Installment 3 Date'
]

for col in date_columns:
    course[col] = course[col].astype('string')

In [ ]:
course.dtypes

In [ ]:
course.duplicated().sum()

In [ ]:
course['Name'].duplicated().sum()

In [ ]:
course[course.duplicated(keep=False)].sort_values('Name')

In [ ]:
course = course.dropna(
    subset=['Name']
).reset_index(drop=True)

In [ ]:
course.shape

In [ ]:
course.duplicated().sum()

In [ ]:
course[
    course['Name'].duplicated(keep=False)
].sort_values('Name')

In [ ]:
print("Internship:", internship.columns.tolist())
print("Registered Students:", registered_students.columns.tolist())
print("Payment Entries:", payment_entries.columns.tolist())
print("Student Summary:", student_summary.columns.tolist())
print("Registered Intern Details:", registered_intern_details.columns.tolist())
print("Course:", course.columns.tolist())

In [ ]:
registered_students['Student ID'].equals(student_summary['Student ID'])

In [ ]:
set(registered_students['Student ID']) - set(student_summary['Student ID'])

## 8. Match records across worksheets

Compare student identifiers and names; build matching keys and validate joins.

In [ ]:
set(student_summary['Student ID']) - set(registered_students['Student ID'])

In [ ]:
student_master = registered_students.merge(
    student_summary,
    on='Student ID',
    how='left',
    suffixes=('', '_Summary')
)

In [ ]:
student_master.shape

In [ ]:
student_master.columns.tolist()

In [ ]:
registered_students[['Student ID', 'Student Name']].merge(
    student_summary[['Student ID', 'Student Name']],
    on='Student ID',
    suffixes=('_Registered', '_Summary')
)

In [ ]:
name_check = registered_students[['Student ID', 'Student Name']].merge(
    student_summary[['Student ID', 'Student Name']],
    on='Student ID',
    suffixes=('_Registered', '_Summary')
)

name_check[
    name_check['Student Name_Registered'] != name_check['Student Name_Summary']
]

In [ ]:
registered_students['Student Name'] = (
    registered_students['Student Name']
    .str.strip()
)

student_summary['Student Name'] = (
    student_summary['Student Name']
    .str.strip()
)

In [ ]:
registered_students['Student Name'] = (
    registered_students['Student Name']
    .str.strip()
)

student_summary['Student Name'] = (
    student_summary['Student Name']
    .str.strip()
)

In [ ]:
student_master[
    [
        'Student ID',
        'Total Billing',
        'Total Billing_Summary',
        'Total Collection',
        'Total Collection_Summary',
        'Pending Amount',
        'Pending'
    ]
]

In [ ]:
student_master = student_master.drop(
    columns=[
        'Student Name_Summary',
        'Total Billing_Summary',
        'Total Collection_Summary',
        'Pending'
    ]
)

In [ ]:
student_master.shape

In [ ]:
student_master['Student Name'].isin(
    internship['Name']
).sum()

In [ ]:
student_master['Student Name'].isin(
    registered_intern_details['Name']
).sum()

In [ ]:
student_master['Student Name'].isin(
    registered_intern_details['Name']
).sum()

In [ ]:
student_master[
    student_master['Student Name'].isin(
        internship['Name']
    )
][['Student ID', 'Student Name']]

In [ ]:
student_master[
    student_master['Student Name'].isin(
        registered_intern_details['Name']
    )
][['Student ID', 'Student Name']]

In [ ]:
internship[
    internship['Name'].isin(
        ['Mahalakshmi', 'Keerthana', 'Karthika']
    )
][
    ['S.No', 'DOJ', 'Name', 'Phone number', 'College', 'Duration']
]

In [ ]:
registered_intern_details[
    registered_intern_details['Name'].isin(
        ['Mahalakshmi', 'Keerthana', 'Karthika']
    )
][
    ['S.No', 'Date', 'Name', 'College', 'Duration', 'Group No']
]

In [ ]:
internship['Name_key'] = internship['Name'].str.strip().str.lower()
internship['College_key'] = internship['College'].str.strip().str.lower()
internship['Duration_key'] = internship['Duration'].str.strip().str.lower()

registered_intern_details['Name_key'] = registered_intern_details['Name'].str.strip().str.lower()
registered_intern_details['College_key'] = registered_intern_details['College'].str.strip().str.lower()
registered_intern_details['Duration_key'] = registered_intern_details['Duration'].str.strip().str.lower()

In [ ]:
internship['Date_key'] = pd.to_datetime(
    internship['DOJ'],
    errors='coerce'
)

registered_intern_details['Date_key'] = pd.to_datetime(
    registered_intern_details['Date'],
    errors='coerce'
)

In [ ]:
internship['match_key'] = (
    internship['Name_key'] + '|' +
    internship['College_key'] + '|' +
    internship['Duration_key'] + '|' +
    internship['Date_key'].astype('string')
)

In [ ]:
registered_intern_details['match_key'] = (
    registered_intern_details['Name_key'] + '|' +
    registered_intern_details['College_key'] + '|' +
    registered_intern_details['Duration_key'] + '|' +
    registered_intern_details['Date_key'].astype('string')
)

In [ ]:
internship['match_key'].isin(
    registered_intern_details['match_key']
).sum()

In [ ]:
registered_intern_details[
    ~registered_intern_details['match_key'].isin(
        internship['match_key']
    )
][[
    'S.No',
    'Date',
    'Name',
    'College',
    'Duration',
    'DOMAIN',
    'Certificate Status'
]]

In [ ]:
internship_master = internship.merge(
    registered_intern_details[
        [
            'match_key',
            'DOMAIN',
            'Preferred Start Date',
            'Preferred Time Slot',
            'Assigned Batch Code',
            'Batch Status',
            'Certificate Status'
        ]
    ],
    on='match_key',
    how='left'
)

In [ ]:
kishore = registered_intern_details[
    ~registered_intern_details['match_key'].isin(
        internship['match_key']
    )
]

In [ ]:
kishore[['Name', 'College', 'Duration', 'DOMAIN']]

In [ ]:
internship_master.shape

In [ ]:
internship_master.columns.tolist()

In [ ]:
internship_master[
    ['Name', 'Certificate Status_x', 'Certificate Status_y']
].dropna(
    subset=['Certificate Status_x', 'Certificate Status_y'],
    how='all'
).head(20)

In [ ]:
(
    internship_master['Certificate Status_x'].fillna('MISSING')
    ==
    internship_master['Certificate Status_y'].fillna('MISSING')
).sum()

In [ ]:
conflicts = internship_master[
    internship_master['Certificate Status_x'].notna() &
    internship_master['Certificate Status_y'].notna() &
    (
        internship_master['Certificate Status_y']
        .str.strip()
        .str.lower()
        .map({'yes': 'issued', 'no': 'not issued'})
        !=
        internship_master['Certificate Status_x']
        .str.strip()
        .str.lower()
    )
]

conflicts[['Name', 'Certificate Status_x', 'Certificate Status_y']]

In [ ]:
internship_master['Certificate Status'] = (
    internship_master['Certificate Status_x']
)

In [ ]:
internship_master = internship_master.drop(
    columns=[
        'Certificate Status_x',
        'Certificate Status_y',
        'Name_key',
        'College_key',
        'Duration_key',
        'Date_key',
        'match_key'
    ]
)

In [ ]:
certificate_conflicts = internship_master[
    internship_master['Certificate Status_x'].notna() &
    internship_master['Certificate Status_y'].notna() &
    (internship_master['Certificate Status_x'].str.lower() == 'issued') &
    (internship_master['Certificate Status_y'].str.lower() == 'no')
][[
    'Name',
    'Certificate Status_x',
    'Certificate Status_y'
]]

In [ ]:
internship_master.columns.tolist()

In [ ]:
internship_master[
    internship_master['Name'].str.contains(
        'Kishore', case=False, na=False
    )
][['S.No', 'DOJ', 'Name', 'College', 'Duration', 'DOMAIN']]

In [ ]:
# Privacy note: the original cell inspected or corrected named individual records.
# This record-specific operation was omitted from the public copy.
# Apply approved corrections locally using a documented, non-public exception list.

In [ ]:
internship_master.shape

In [ ]:
print("Internship:", len(internship))
print("Registered Intern Details:", len(registered_intern_details))
print("Internship Master:", len(internship_master))

In [ ]:
registered_intern_details[
    registered_intern_details['match_key'].duplicated(keep=False)
][
    ['match_key', 'S.No', 'Date', 'Name', 'College', 'Duration']
].sort_values('match_key')

In [ ]:
internship_master = internship.merge(
    registered_intern_details[
        [
            'match_key',
            'DOMAIN',
            'Preferred Start Date',
            'Preferred Time Slot',
            'Assigned Batch Code',
            'Batch Status',
            'Certificate Status'
        ]
    ][
        registered_intern_details['match_key'].notna()
    ],
    on='match_key',
    how='left'
)

In [ ]:
internship_master.shape

In [ ]:
unmatched = registered_intern_details[
    ~registered_intern_details['match_key'].isin(
        internship[internship['match_key'].notna()]['match_key']
    )
]

unmatched[
    ['S.No', 'Date', 'Name', 'College', 'Duration', 'DOMAIN']
]

In [ ]:
internship['match'] = (
    internship['Name'].str.strip().str.lower() + '|' +
    internship['College'].str.strip().str.lower()
)

registered_intern_details['match'] = (
    registered_intern_details['Name'].str.strip().str.lower() + '|' +
    registered_intern_details['College'].str.strip().str.lower()
)

In [ ]:
internship['match'].isin(
    registered_intern_details['match']
).sum()

In [ ]:
internship[
    ~internship['match'].isin(
        registered_intern_details['match']
    )
][['S.No', 'DOJ', 'Name', 'College', 'Duration']]

In [ ]:
registered_intern_details = registered_intern_details.rename(
    columns={'Date': 'DOJ'}
)

In [ ]:
registered_intern_details = registered_intern_details.reindex(
    columns=internship.columns
)

## 9. Build and refine the consolidated master table

Combine cleaned sources and fill missing course/domain information.

In [ ]:
combined_internship = pd.concat(
    [internship, registered_intern_details],
    ignore_index=True
)

In [ ]:
combined_internship.head()

In [ ]:
combined_internship.shape

In [ ]:
internship = pd.read_excel(
    file_path,
    sheet_name="Internship",
    header=3
)

In [ ]:
internship.shape

In [ ]:
internship_master.shape

In [ ]:
internship_master.columns.tolist()

In [ ]:
internship_master.head()

In [ ]:
internship_master['Certificate Status'] = (
    internship_master['Certificate Status_x']
)

In [ ]:
internship_master = internship_master.drop(
    columns=[
        'Certificate Status_x',
        'Certificate Status_y',
        'Name_key',
        'College_key',
        'Duration_key',
        'Date_key',
        'match_key'
    ]
)

In [ ]:
internship_master.shape

In [ ]:
course['match_key'] = (
    course['Name'].astype('string').str.strip().str.lower() + '|' +
    course['Program'].astype('string').str.strip().str.lower() + '|' +
    course['DOJ'].astype('string')
)

internship_master['match_key'] = (
    internship_master['Name'].astype('string').str.strip().str.lower() + '|' +
    internship_master['Program'].astype('string').str.strip().str.lower() + '|' +
    internship_master['DOJ'].astype('string')
)

In [ ]:
course['match_key'] = (
    course['Name'].astype('string').str.strip().str.lower() + '|' +
    course['DOJ'].astype('string')
)

internship_master['match_key'] = (
    internship_master['Name'].astype('string').str.strip().str.lower() + '|' +
    internship_master['DOJ'].astype('string')
)

In [ ]:
course['match_key'].isin(
    internship_master['match_key']
).sum()

In [ ]:
course[['Name', 'DOJ', 'Program', 'Total Billing']].head(20)

In [ ]:
course.columns.tolist()

In [ ]:
internship_master.columns.tolist()

In [ ]:
course = course.drop(columns='match_key')
internship_master = internship_master.drop(columns='match_key')

In [ ]:
final_master = pd.concat(
    [internship_master, course],
    ignore_index=True,
    sort=False
)

In [ ]:
final_master.shape

In [ ]:
print("internship_master:", internship_master.shape)
print("course:", course.shape)
print("final_master:", final_master.shape)

In [ ]:
print("internship_master columns:")
print(internship_master.columns.tolist())

print("\ncourse columns:")
print(course.columns.tolist())

In [ ]:
# Privacy note: the original cell inspected or corrected named individual records.
# This record-specific operation was omitted from the public copy.
# Apply approved corrections locally using a documented, non-public exception list.

In [ ]:
internship_master.shape

In [ ]:
course = course.rename(columns={
    'Total Billing': 'Total billing',
    'Total Collection': 'Total collection',
    'Pending Amount': 'Pending amount'
})

In [ ]:
course.columns.tolist()

In [ ]:
course = course.reindex(columns=internship_master.columns)

In [ ]:
final_master = pd.concat(
    [internship_master, course],
    ignore_index=True
)

In [ ]:
final_master.shape

In [ ]:
final_master.info()

In [ ]:
final_master.isna().sum()

In [ ]:
course.columns.tolist()

In [ ]:
final_master['Name'].duplicated().sum()

In [ ]:
final_master[
    final_master['Name'].duplicated(keep=False)
][['Name', 'DOJ', 'College', 'Duration']].sort_values('Name')

In [ ]:
final_master

In [ ]:
course['Name'].duplicated().sum()

In [ ]:
program_map = dict(zip(course['Name'], course['Program']))

final_master['DOMAIN'] = final_master['DOMAIN'].fillna(
    final_master['Name'].map(program_map)
)

In [ ]:
course_original.columns.tolist()

In [ ]:
[name for name in globals() if 'course' in name.lower()]

In [ ]:
course_original = pd.read_excel(
    file_path,
    sheet_name="Course",
    header=3
)

In [ ]:
course_original.columns.tolist()

In [ ]:
course_original.shape

In [ ]:
course_original[
    course_original['Name'].notna()
][['S.No', 'Name', 'Program']]

In [ ]:
course_program = (
    course_original[['Name', 'Program']]
    .dropna(subset=['Name'])
    .copy()
)

course_program['Name'] = (
    course_program['Name']
    .astype('string')
    .str.strip()
    .str.lower()
)

In [ ]:
master_name = (
    final_master['Name']
    .astype('string')
    .str.strip()
    .str.lower()
)

In [ ]:
program_map = dict(
    zip(course_program['Name'], course_program['Program'])
)

final_master['DOMAIN'] = final_master['DOMAIN'].fillna(
    master_name.map(program_map)
)

In [ ]:
final_master['DOMAIN'].notna().sum()

In [ ]:
course_program[
    ~course_program['Name'].isin(
        master_name[final_master['DOMAIN'].notna()]
    )
]

In [ ]:
course_program['matched'] = course_program['Name'].isin(
    master_name
)

course_program[['Name', 'Program', 'matched']]

In [ ]:
final_master[
    final_master['Name'].str.strip().str.lower().isin(
        course_program['Name']
    )
][['Name', 'DOJ', 'College', 'Duration', 'DOMAIN']]

In [ ]:
final_master['Name'].isin(
    student_master['Student Name']
).sum()

In [ ]:
final_master[
    final_master['Name'].isin(
        student_master['Student Name']
    )
][[
    'Name',
    'DOJ',
    'College',
    'Duration',
    'DOMAIN'
]]

In [ ]:
student_master[
    student_master['Student Name'].isin(
        ['Karthika', 'Keerthana', 'Mahalakshmi']
    )
][[
    'Student ID',
    'Student Name',
    'College',
    'Duration',
    'DOJ'
]]

In [ ]:
print("final_master:")
print(final_master.columns.tolist())

print("\nstudent_master:")
print(student_master.columns.tolist())

In [ ]:
final_master['Student ID'] = pd.NA

In [ ]:
student_master = student_master.rename(columns={
    'Student Name': 'Name',
    'Phone Number': 'Phone number',
    'Total Billing': 'Total billing',
    'Total Collection': 'Total collection',
    'Pending Amount': 'Pending amount'
})

In [ ]:
student_master = student_master.rename(
    columns={'Program': 'DOMAIN'}
)

In [ ]:
student_master = student_master.reindex(
    columns=final_master.columns
)

In [ ]:
final_master = pd.concat(
    [final_master, student_master],
    ignore_index=True
)

In [ ]:
final_master.shape

In [ ]:
final_master.info()

In [ ]:
final_master.isna().sum()

In [ ]:
final_master[
    final_master['Total billing'].isna()
][[
    'Name',
    'DOJ',
    'College',
    'Duration',
    'DOMAIN',
    'Total billing',
    'Total collection',
    'Pending amount'
]]

In [ ]:
course_original[
    course_original['Name'].astype('string').str.strip().str.lower() == 'saran'
][['S.No', 'Name', 'Program', 'DOJ', 'Total billing']]

In [ ]:
final_master.loc[
    final_master['Name'].eq('Saran'),
    'DOJ'
] = pd.NaT

In [ ]:
final_master[
    final_master['Name'].eq('Saran')
][['Name', 'DOJ', 'DOMAIN', 'Total billing']]

In [ ]:
final_master.loc[
    final_master['Name'].eq('Saran'),
    'Total billing'
] = 35000

In [ ]:
final_master.loc[
    final_master['Name'].eq('Saran'),
    'DOJ'
] = pd.NaT

In [ ]:
final_master[
    final_master['Name'].eq('Saran')
][['Name', 'DOJ', 'DOMAIN', 'Total billing']]

In [ ]:
# Privacy note: the original cell inspected or corrected named individual records.
# This record-specific operation was omitted from the public copy.
# Apply approved corrections locally using a documented, non-public exception list.

In [ ]:
# Privacy note: the original cell inspected or corrected named individual records.
# This record-specific operation was omitted from the public copy.
# Apply approved corrections locally using a documented, non-public exception list.

## 10. Reconcile billing, collections, and pending balances

Compare billed amounts, payment totals, installment values, and outstanding balances.

In [ ]:
payment_entries['Student ID'].isin(
    student_master['Student ID']
).value_counts()

In [ ]:
payment_entries[
    ~payment_entries['Student ID'].isin(
        student_master['Student ID']
    )
][['Student ID', 'Student Name', 'Amount']]

In [ ]:
payment_entries.groupby('Student ID')['Amount'].sum()

In [ ]:
payment_totals = (
    payment_entries
    .groupby('Student ID')['Amount']
    .sum()
    .reset_index(name='Payment Total')
)

comparison = student_master[
    ['Student ID', 'Student Name', 'Total Collection']
].merge(
    payment_totals,
    on='Student ID',
    how='left'
)

comparison['Difference'] = (
    comparison['Total Collection'] -
    comparison['Payment Total']
)

comparison

In [ ]:
payment_totals = (
    payment_entries
    .groupby('Student ID')['Amount']
    .sum()
    .reset_index(name='Payment Total')
)

comparison = student_master[
    ['Student ID', 'Name', 'Total collection']
].merge(
    payment_totals,
    on='Student ID',
    how='left'
)

comparison['Difference'] = (
    comparison['Total collection'] -
    comparison['Payment Total']
)

comparison

In [ ]:
comparison[comparison['Difference'] != 0]

In [ ]:
payment_totals = (
    payment_entries
    .groupby('Student ID')['Amount']
    .sum()
    .reset_index(name='Payment Total')
)

comparison = student_summary[
    ['Student ID', 'Student Name', 'Total Collection']
].merge(
    payment_totals,
    on='Student ID',
    how='left'
)

comparison['Difference'] = (
    comparison['Total Collection'] -
    comparison['Payment Total']
)

comparison

In [ ]:
comparison[comparison['Difference'] != 0]

In [ ]:
payment_entries[
    payment_entries['Student Name'].isin(
        final_master['Name']
    )
][['Student ID', 'Student Name']].drop_duplicates()

In [ ]:
payment_entries[
    payment_entries['Student Name'].isin(
        final_master['Name']
    )
]['Student Name'].nunique()

In [ ]:
payment_names = (
    payment_entries['Student Name']
    .dropna()
    .unique()
)

payment_name_matches = final_master[
    final_master['Name'].isin(payment_names)
][
    ['Name', 'DOJ', 'College', 'Duration', 'DOMAIN', 'Total billing']
].sort_values('Name')

payment_name_matches

In [ ]:
payment_student_info = student_master[
    student_master['Student ID'].isin(
        payment_entries['Student ID']
    )
][[
    'Student ID',
    'Name',
    'DOJ',
    'College',
    'Duration'
]].copy()

payment_student_info

In [ ]:
# Create normalised matching keys
final_master['_student_match_key'] = (
    final_master['Name'].astype('string').str.strip().str.lower() + '|' +
    final_master['DOJ'].astype('string') + '|' +
    final_master['Duration'].astype('string').str.strip().str.lower()
)

student_master['_student_match_key'] = (
    student_master['Name'].astype('string').str.strip().str.lower() + '|' +
    student_master['DOJ'].astype('string') + '|' +
    student_master['Duration'].astype('string').str.strip().str.lower()
)

In [ ]:
student_id_map = dict(
    zip(
        student_master['_student_match_key'],
        student_master['Student ID']
    )
)

In [ ]:
final_master['Student ID'] = final_master['Student ID'].fillna(
    final_master['_student_match_key'].map(student_id_map)
)

In [ ]:
final_master.drop(
    columns=['_student_match_key'],
    inplace=True
)

student_master.drop(
    columns=['_student_match_key'],
    inplace=True
)

In [ ]:
final_master['Student ID'].notna().sum()

In [ ]:
final_master[
    final_master['Student ID'].notna()
][[
    'Student ID',
    'Name',
    'DOJ',
    'College',
    'Duration'
]].sort_values('Student ID')

In [ ]:
final_master[
    final_master['Student ID'].eq('IN0571')
].T

In [ ]:
final_master = final_master.drop(index=544).reset_index(drop=True)

In [ ]:
final_master.shape

In [ ]:
final_master[
    final_master['Student ID'].eq('IN0571')
][['Student ID', 'Name', 'DOJ', 'Phone number',
   'College', 'Duration', 'Total billing']]

In [ ]:
final_master[
    final_master['Student ID'].duplicated(keep=False)
][[
    'Student ID',
    'Name',
    'DOJ',
    'College',
    'Duration'
]].sort_values('Student ID')

In [ ]:
duplicate_student_ids = final_master[
    final_master['Student ID'].notna() &
    final_master['Student ID'].duplicated(keep=False)
][[
    'Student ID',
    'Name',
    'DOJ',
    'College',
    'Duration'
]].sort_values('Student ID')

duplicate_student_ids

In [ ]:
final_master['Student ID'].dropna().nunique()

In [ ]:
payment_entries['Student ID'].nunique()

In [ ]:
final_master[
    final_master['Name'].astype('string').str.contains(
        'kiruth', case=False, na=False
    )
][[
    'Name',
    'DOJ',
    'College',
    'Duration',
    'DOMAIN',
    'Student ID'
]]

In [ ]:
payment_final_match = payment_entries[
    payment_entries['Student ID'].isin(
        final_master['Student ID'].dropna()
    )
][[
    'Student ID',
    'Student Name'
]].drop_duplicates().merge(
    final_master[
        final_master['Student ID'].notna()
    ][[
        'Student ID',
        'Name',
        'College',
        'DOJ',
        'Duration',
        'DOMAIN'
    ]],
    on='Student ID',
    how='left'
)

payment_final_match

In [ ]:
payment_final_match = payment_entries[
    payment_entries['Student ID'].isin(
        final_master['Student ID'].dropna()
    )
][[
    'Student ID',
    'Student Name'
]].drop_duplicates().merge(
    final_master[
        final_master['Student ID'].notna()
    ][[
        'Student ID',
        'Name',
        'Phone number',
        'College',
        'DOJ',
        'Duration',
        'DOMAIN'
    ]],
    on='Student ID',
    how='left'
)

payment_final_match[
    ['Student ID', 'Student Name', 'Phone number', 'College']
]

In [ ]:
final_master[
    final_master['Student ID'].eq('IN0554')
].T

In [ ]:
final_master[
    final_master['Student ID'].eq('IN0554')
][[
    'Student ID',
    'Name',
    'DOJ',
    'Phone number',
    'College',
    'Duration',
    'Total billing',
    'Total collection',
    'Pending amount'
]]

In [ ]:
payment_ids = set(payment_entries['Student ID'].dropna())
master_ids = set(final_master['Student ID'].dropna())

unmatched_payment = payment_ids - master_ids
unmatched_master = master_ids - payment_ids

print("Payment Entries not in final_master:", len(unmatched_payment))
print("final_master not in Payment Entries:", len(unmatched_master))

print("Payment only:", unmatched_payment)
print("Master only:", unmatched_master)

In [ ]:
payment_summary = (
    payment_entries
    .groupby('Student ID', as_index=False)
    .agg(
        Payment_Total=('Amount', 'sum'),
        Payment_Count=('Amount', 'count')
    )
)

In [ ]:
final_master = final_master.merge(
    payment_summary,
    on='Student ID',
    how='left'
)

In [ ]:
final_master[
    final_master['Student ID'].eq('IN0554')
].T

In [ ]:
payment_summary = (
    payment_entries
    .groupby('Student ID', as_index=False)
    .agg(
        Payment_Total=('Amount', 'sum'),
        Payment_Count=('Amount', 'count')
    )
)

In [ ]:
final_master = final_master.merge(
    payment_summary,
    on='Student ID',
    how='left'
)

In [ ]:
final_master.shape

In [ ]:
final_master[
    final_master['Student ID'].eq('IN0554')
][['Student ID', 'Name', 'Payment_Total', 'Payment_Count']]

In [ ]:
payment_summary = (
    payment_entries
    .groupby('Student ID', as_index=False)
    .agg(
        Payment_Total=('Amount', 'sum'),
        Payment_Count=('Amount', 'count')
    )
)

In [ ]:
payment_summary.head()

In [ ]:
final_master = final_master.merge(
    payment_summary,
    on='Student ID',
    how='left'
)

In [ ]:
final_master.columns.tolist()

In [ ]:
final_master[
    final_master['Student ID'].eq('IN0554')
][[
    'Student ID',
    'Name',
    'Payment_Total',
    'Payment_Count'
]]

In [ ]:
payment_matched_master = final_master[
    final_master['Student ID'].isin(
        payment_entries['Student ID'].dropna().unique()
    )
].copy()

In [ ]:
payment_matched_master

In [ ]:
payment_matched_master[
    [
        'Student ID',
        'Name',
        'Phone number',
        'College',
        'DOJ',
        'Duration',
        'DOMAIN',
        'Total billing',
        'Payment_Total',
        'Payment_Count'
    ]
]

In [ ]:
final_master.columns

In [ ]:
final_master.dtypes

In [ ]:
final_master.isna().sum().sort_values(ascending=False)

In [ ]:
final_master.duplicated().sum()

In [ ]:
final_master['Student ID'].duplicated().sum()

In [ ]:
final_master['Student ID'].dropna().duplicated().sum()

In [ ]:
final_master['Name'].duplicated().sum()

In [ ]:
final_master[final_master['Name'].duplicated(keep=False)] \
    .sort_values('Name')[['Name', 'Phone number', 'College', 'DOJ', 'Student ID']]

In [ ]:
final_master['Phone number'].dropna().duplicated().sum()

In [ ]:
final_master[
    final_master['Phone number'].notna() &
    final_master['Phone number'].duplicated(keep=False)
][['Name', 'Phone number', 'College', 'DOJ', 'Student ID']].sort_values('Phone number')

In [ ]:
final_master['Phone number'].value_counts(dropna=False).head(20)

In [ ]:
final_master['Phone number'].dropna()

In [ ]:
(final_master == 'nan').sum().sort_values(ascending=False)

In [ ]:
final_master['Student ID'].value_counts(dropna=False)

In [ ]:
final_master['S.No'].value_counts(dropna=False).sort_index()

In [ ]:
final_master[
    final_master['S.No'].duplicated(keep=False)
][['S.No', 'Name', 'Phone number', 'College', 'DOJ', 'Student ID']].sort_values('S.No')

In [ ]:
final_master['S.No'] = range(1, len(final_master) + 1)

In [ ]:
final_master['DOJ'].agg(['min', 'max'])

In [ ]:
final_master[final_master['DOJ'] > pd.Timestamp.today()][
    ['S.No', 'Name', 'DOJ', 'College', 'Phone number', 'Student ID']
]

In [ ]:
final_master[
    final_master['Name'].isin(['Rajkumar', 'Josuah'])
][['S.No', 'Name', 'DOJ', 'College', 'Phone number', 'Student ID']]

In [ ]:
final_master.loc[
    final_master['Name'].isin(['Rajkumar', 'Josuah']),
    'DOJ'
] = pd.Timestamp('2026-05-01')

In [ ]:
final_master[
    final_master['Name'].isin(['Rajkumar', 'Josuah'])
][['Name', 'DOJ']]

In [ ]:
final_master['Phone number'] = final_master['Phone number'].replace('nan', pd.NA)

In [ ]:
final_master['Phone number'].value_counts(dropna=False).head()

In [ ]:
final_master['Phone number'] = (
    final_master['Phone number']
    .replace(['nan', 'NaN'], pd.NA)
)

In [ ]:
final_master['Phone number'].isna().sum()

In [ ]:
final_master['College'].value_counts(dropna=False)

In [ ]:
college_case_check = (
    final_master['College']
    .dropna()
    .str.strip()
    .str.lower()
    .value_counts()
)

college_case_check

In [ ]:
college_clean = final_master['College'].dropna().str.strip().str.lower()

duplicates = (
    pd.DataFrame({
        'Original': final_master.loc[college_clean.index, 'College'],
        'Normalised': college_clean
    })
    .groupby('Normalised')['Original']
    .agg(lambda x: sorted(x.unique()))
)

duplicates[duplicates.apply(len) > 1]

In [ ]:
college_mapping = {
    'A.V.C college of engineering': 'A.V.C College of Engineering',
    'Adhiparasakthi College of engineering': 'Adhiparasakthi College of Engineering',
    'Dhanalakshmi Srinivasan University': 'Dhanalakshmi Srinivasan University',
    'Jeppiaar college': 'Jeppiaar Engineering College',
    'JEPPIAAR UNIVERSITY': 'Jeppiaar University',
    'PANIMALAR COLLEGE': 'Panimalar Engineering College',
    'Panimalar College': 'Panimalar Engineering College',
    'Panimalar college': 'Panimalar Engineering College',
    'Rajalakshmi engineering college': 'Rajalakshmi Engineering College',
    'Rajalakshmi Engineering College': 'Rajalakshmi Engineering College',
    'RMD ENGINEERING COLLEGE': 'R.M.D. Engineering College',
    'RMD Engineering College': 'R.M.D. Engineering College',
    'RMK college of Engineering and Technology': 'R.M.K. College of Engineering and Technology',
    'R.M.K. College of Engineering and Technology': 'R.M.K. College of Engineering and Technology',
    'Saveetha Engineering college': 'Saveetha Engineering College',
    'saveetha Engineering college': 'Saveetha Engineering College',
    'SMIT': 'Sri Muthukumaran Institute of Technology',
    'Smit': 'Sri Muthukumaran Institute of Technology',
    'Sri Muthukumaran institute of technology': 'Sri Muthukumaran Institute of Technology',
    'Sri Muthukumaran Institute of Technology': 'Sri Muthukumaran Institute of Technology',
    'SRM MADURAI COLLEGE FOR ENGINEERING AND TECHNOLOGY': 'SRM Madurai College for Engineering and Technology',
    'Srm Madurai College for Engineering and Technology': 'SRM Madurai College for Engineering and Technology',
    'T.J.S.ENGINEERING COLLEGE': 'T.J.S. Engineering College',
    'T.J.S.engineering college': 'T.J.S. Engineering College',
    'Vels University': 'Vels University Pallavaram',
    'Vels University ': 'Vels University Pallavaram',
    'Vels university': 'Vels University Pallavaram',
    'Vels university pallavaram': 'Vels University Pallavaram',
    'Vels University Pallavaram': 'Vels University Pallavaram',
    'St. Thomas college': 'St. Thomas College of Arts and Science',
    'St. thomas college': 'St. Thomas College of Arts and Science',
    'Dhanalakshmi college': 'Dhanalakshmi College of Engineering',
    'Dhanalakshmi engineering college': 'Dhanalakshmi College of Engineering'
}

final_master['College'] = final_master['College'].replace(college_mapping)

In [ ]:
final_master['College'].value_counts(dropna=False)

In [ ]:
final_master['College'] = final_master['College'].replace({
    'Adhiparasakthi Engineering college Melmaruvathur': 'Adhiparasakthi College of Engineering',
    'Adhiparasakthi engineering college melmaruvathurr': 'Adhiparasakthi College of Engineering'
})

In [ ]:
college_clean = final_master['College'].dropna().str.strip().str.lower()

remaining_duplicates = (
    pd.DataFrame({
        'Original': final_master.loc[college_clean.index, 'College'],
        'Normalised': college_clean
    })
    .groupby('Normalised')['Original']
    .agg(lambda x: sorted(x.unique()))
)

remaining_duplicates[remaining_duplicates.apply(len) > 1]

In [ ]:
college_mapping_2 = {
    # A.V.C
    'A.V.C college of engineering': 'A.V.C College of Engineering',

    # Adhiparasakthi
    'Adhiparasakthi College of engineering': 'Adhiparasakthi College of Engineering',

    # Dhanalakshmi Srinivasan
    'Dhanalakshmi Srinivasan University': 'Dhanalakshmi Srinivasan University',

    # Jeppiaar University
    'Jeppiaar university': 'Jeppiaar University',

    # Panimalar
    'PANIMALAR ENGINEERING COLLEGE': 'Panimalar Engineering College',
    'Panimalar engineering college': 'Panimalar Engineering College',

    # Rajalakshmi
    'Rajalakshmi engineering college': 'Rajalakshmi Engineering College',

    # RMK
    'RMK college of Engineering and Technology': 'R.M.K. College of Engineering and Technology',

    # Saveetha
    'saveetha Engineering college': 'Saveetha Engineering College',

    # Sri Muthukumaran
    'Sri Muthukumaran institute of technology': 'Sri Muthukumaran Institute of Technology',

    # SRM Madurai
    'SRM MADURAI COLLEGE FOR ENGINEERING AND TECHNOLOGY':
        'SRM Madurai College for Engineering and Technology',

    # TJS
    'T.J.S. Engineering College': 'T.J.S. Engineering College',
    'T.J.S.ENGINEERING COLLEGE': 'T.J.S. Engineering College',
    'T.J.S.engineering college': 'T.J.S. Engineering College',

    # Vels
    'Vels University Pallavaram': 'Vels University Pallavaram',
    'Vels University pallavaram': 'Vels University Pallavaram',
    'Vels university Pallavaram': 'Vels University Pallavaram'
}

final_master['College'] = final_master['College'].replace(college_mapping_2)

In [ ]:
final_master['College'].nunique(dropna=True)

In [ ]:
final_master['College'].value_counts(dropna=False).head(30)

In [ ]:
final_master['College'] = final_master['College'].replace(
    'RMD ENGINEERING COLLEGE',
    'R.M.D. Engineering College'
)

In [ ]:
final_master['College'] = final_master['College'].replace(
    'Panimalar engineering college',
    'Panimalar Engineering College'
)

In [ ]:
final_master['College'].value_counts(dropna=False).head(30)

In [ ]:
final_master[
    final_master['College'].astype('string').str.contains(
        'panimalar', case=False, na=False
    )
]['College'].value_counts(dropna=False)

In [ ]:
panimalar_variants = [
    'Panimalar Engineering College',
    'Panimalar engineering college',
    'Panimalar Engineering collage',
    'Panimalar Engineering college',
    'PANIMALAR ENGINEERING COLLEGE'
]

final_master.loc[
    final_master['College'].isin(panimalar_variants),
    'College'
] = 'Panimalar Engineering College'

In [ ]:
final_master[
    final_master['College'].astype('string').str.contains(
        'panimalar', case=False, na=False
    )
]['College'].value_counts()

In [ ]:
final_master[
    final_master['College'].astype('string').str.contains(
        'panimalar', case=False, na=False
    )
]['College'].map(repr).value_counts()

In [ ]:
final_master['College'] = final_master['College'].str.strip()

In [ ]:
final_master[
    final_master['College'].astype('string').str.contains(
        'panimalar', case=False, na=False
    )
]['College'].value_counts()

In [ ]:
panimalar_mask = (
    final_master['College']
    .astype('string')
    .str.strip()
    .str.lower()
    .str.replace('collage', 'college', regex=False)
    .eq('panimalar engineering college')
)

final_master.loc[panimalar_mask, 'College'] = 'Panimalar Engineering College'

In [ ]:
final_master[
    final_master['College'] == 'Panimalar Engineering College'
].shape[0]

In [ ]:
tjs_mask = (
    final_master['College']
    .astype('string')
    .str.strip()
    .str.lower()
    .isin([
        'tjs college',
        't.j.s. engineering college',
        't.j.s.engineering college'
    ])
)

final_master.loc[tjs_mask, 'College'] = 'T.J.S. Engineering College'

In [ ]:
final_master[
    final_master['College'] == 'T.J.S. Engineering College'
].shape[0]

In [ ]:
final_master['College'] = final_master['College'].replace(
    'Dhanalakshmi srinivasan engineering college peramablur',
    'Dhanalakshmi Srinivasan Engineering College'
)

In [ ]:
final_master[
    final_master['College'] == 'Dhanalakshmi Srinivasan Engineering College'
].shape[0]

In [ ]:
final_master['College'] = final_master['College'].replace(
    'Adhi college',
    'Adhi College of Engineering and Technology'
)

In [ ]:
final_master[
    final_master['College'] == 'Adhi College of Engineering and Technology'
].shape[0]

In [ ]:
final_master['College'] = final_master['College'].replace(
    'Gtech college',
    "Ganadipathy Tulsi's Jain Engineering College"
)

In [ ]:
final_master[
    final_master['College'] == "Ganadipathy Tulsi's Jain Engineering College"
].shape[0]

In [ ]:
final_master['College'] = final_master['College'].replace(
    'RMK College',
    'R.M.K. College of Engineering and Technology'
)

In [ ]:
final_master[
    final_master['College'] == 'R.M.K. College of Engineering and Technology'
].shape[0]

In [ ]:
final_master['College'] = final_master['College'].replace(
    'Gtech college',
    "Ganadipathy Tulsi's Jain Engineering College"
)

In [ ]:
final_master[
    final_master['College'] == "Ganadipathy Tulsi's Jain Engineering College"
].shape[0]

In [ ]:
print("Test")

In [ ]:
final_master.to_pickle("final_master_backup.pkl")

In [ ]:
final_master['College'] = final_master['College'].replace(
    'SRM',
    'SRM College of Engineering and Technology'
)

In [ ]:
final_master[
    final_master['College'] == 'SRM College of Engineering and Technology'
].shape[0]

In [ ]:
final_master['College'] = final_master['College'].replace(
    'SRM College',
    'SRM College of Engineering and Technology'
)

In [ ]:
final_master[
    final_master['College'] == 'SRM College of Engineering and Technology'
].shape[0]

In [ ]:
final_master[
    final_master['College'] == 'SRM College of Engineering and Technology'
].shape[0]

In [ ]:
college_mapping = {
    # Confirmed earlier
    'A.V.C college of engineering': 'A.V.C College of Engineering',
    'A.V.C College of engineering': 'A.V.C College of Engineering',

    'Adhiparasakthi College of engineering': 'Adhiparasakthi College of Engineering',
    'Adhiparasakthi Engineering college Melmaruvathur': 'Adhiparasakthi College of Engineering',
    'Adhiparasakthi engineering college melmaruvathurr': 'Adhiparasakthi College of Engineering',

    'Dhanalakshmi Srinivasan University': 'Dhanalakshmi Srinivasan University',

    'Jeppiaar college': 'Jeppiaar Engineering College',
    'Jeppiaar college ': 'Jeppiaar Engineering College',

    'JEPPIAAR UNIVERSITY': 'Jeppiaar University',
    'Jeppiaar university': 'Jeppiaar University',

    'Panimalar College': 'Panimalar Engineering College',
    'Panimalar college': 'Panimalar Engineering College',
    'PANIMALAR COLLEGE': 'Panimalar Engineering College',
    'PANIMALAR ENGINEERING COLLEGE': 'Panimalar Engineering College',
    'Panimalar engineering college': 'Panimalar Engineering College',
    'Panimalar Engineering college': 'Panimalar Engineering College',
    'Panimalar Engineering collage': 'Panimalar Engineering College',

    'Rajalakshmi engineering college': 'Rajalakshmi Engineering College',

    'RMD ENGINEERING COLLEGE': 'R.M.D. Engineering College',
    'RMD Engineering College': 'R.M.D. Engineering College',

    'RMK College': 'R.M.K. College of Engineering and Technology',
    'RMK college of Engineering and Technology': 'R.M.K. College of Engineering and Technology',
    'R.M.K. College of Engineering and Technology': 'R.M.K. College of Engineering and Technology',

    'Saveetha Engineering college': 'Saveetha Engineering College',
    'saveetha Engineering college': 'Saveetha Engineering College',

    'SMIT': 'Sri Muthukumaran Institute of Technology',
    'Smit': 'Sri Muthukumaran Institute of Technology',
    'Sri Muthukumaran institute of technology': 'Sri Muthukumaran Institute of Technology',

    'SRM': 'SRM College of Engineering and Technology',
    'SRM College': 'SRM College of Engineering and Technology',

    'SRM MADURAI COLLEGE FOR ENGINEERING AND TECHNOLOGY':
        'SRM Madurai College for Engineering and Technology',
    'Srm Madurai College for Engineering and Technology':
        'SRM Madurai College for Engineering and Technology',

    'TJS College': 'T.J.S. Engineering College',
    'T.J.S.ENGINEERING COLLEGE': 'T.J.S. Engineering College',
    'T.J.S.engineering college': 'T.J.S. Engineering College',

    'Vels University': 'Vels University Pallavaram',
    'Vels university': 'Vels University Pallavaram',
    'Vels university pallavaram': 'Vels University Pallavaram',
    'Vels University Pallavaram': 'Vels University Pallavaram',

    'St. Thomas college': 'St. Thomas College of Arts and Science',
    'St. thomas college': 'St. Thomas College of Arts and Science',
    'St.Thomas College': 'St. Thomas College of Arts and Science',

    'Dhanalakshmi college': 'Dhanalakshmi College of Engineering',
    'Dhanalakshmi engineering college': 'Dhanalakshmi College of Engineering',

    'Dhanalakshmi srinivasan engineering college peramablur':
        'Dhanalakshmi Srinivasan Engineering College',

    'Adhi college': 'Adhi College of Engineering and Technology',

    'Gtech college': "Ganadipathy Tulsi's Jain Engineering College",
}

In [ ]:
final_master['College'] = final_master['College'].replace(college_mapping)

In [ ]:
final_master['College'] = final_master['College'].str.strip()

In [ ]:
final_master['College'].value_counts(dropna=False).head(40)

In [ ]:
sorted(final_master['College'].dropna().unique())

In [ ]:
college_mapping = {

    # A.V.C
    'A. V. C college of engineering': 'A.V.C College of Engineering',
    'A.V .C COLLEGE OF ENGINEERING': 'A.V.C College of Engineering',
    'A.V.C College of Engineering': 'A.V.C College of Engineering',

    # Adhiparasakthi Engineering College – Melmaruvathur
    'ADHIPARASAKTHI ENGINEERING COLLEGE, MELMARUVATHUR':
        'Adhiparasakthi Engineering College',
    'Aadhiparasakthi Eng College':
        'Adhiparasakthi Engineering College',
    'Adhiparasakthi college of Engineering':
        'Adhiparasakthi Engineering College',
    'Adhiparasakthi college of engineering':
        'Adhiparasakthi Engineering College',

    # Adhi
    'Adhi college': 'Adhi College of Engineering and Technology',

    # AAA
    'AAA Engineering college': 'AAA College of Engineering and Technology',

    # ANJAC
    'ANJAC': 'Ayya Nadar Janaki Ammal College',

    # AMJ
    'Amj': 'Agurchand Manmull Jain College',
    'AMJ': 'Agurchand Manmull Jain College',

    # Anna Adarsh
    'Anna Adarsh College': 'Anna Adarsh College for Women',

    # Annamalai
    'Annamalai University': 'Annamalai University',

    # Auxillium
    'Auxillium college': 'Auxilium College',

    # Avichi
    'Avichi college': 'Avichi College of Arts and Science',

    # CTTE
    'CTTE College': 'CTTE College for Women',

    # Chenduran
    'Chenduran college of Engineering and Technology':
        'Chenduran College of Engineering and Technology',

    # Dhanalakshmi
    'Dhanalakshmi College of Engineering':
        'Dhanalakshmi College of Engineering',
    'Dhanalakshmi Engineering college':
        'Dhanalakshmi College of Engineering',
    'Dhanalakshmi Srinivasa college':
        'Dhanalakshmi Srinivasan College',
    'Dhanalakshmi Srinivasan Engineering College':
        'Dhanalakshmi Srinivasan Engineering College',
    'Dhanalakshmi Srinivasan Engineering College (A), Perambalur':
        'Dhanalakshmi Srinivasan Engineering College',
    'Dhanalakshmi Srinivasan Engineering College(A), Perambalur':
        'Dhanalakshmi Srinivasan Engineering College',
    'Dhanalakshmi Srinivasan Engineering College, Perambalur':
        'Dhanalakshmi Srinivasan Engineering College',
    'Dhanalakshmi Srinivasan Engineering college':
        'Dhanalakshmi Srinivasan Engineering College',
    'Dhanalakshmi Srinivasan engineering collge (A) Perambalur':
        'Dhanalakshmi Srinivasan Engineering College',
    'Dhanalakshmi srinivasan engineering college (autonomous)':
        'Dhanalakshmi Srinivasan Engineering College',

    # Dhanalakshmi Srinivasan University
    'Dhanalakshmi Srinivasan University':
        'Dhanalakshmi Srinivasan University',
    'Dhanalakshmi Srinivasan university':
        'Dhanalakshmi Srinivasan University',
    'Dhanalakshmi srinivasan university':
        'Dhanalakshmi Srinivasan University',
    'Dhanalakshmi srinivasan university trichy':
        'Dhanalakshmi Srinivasan University',

    # Don Bosco
    'Don bosco college': 'Don Bosco College',

    # Eswari
    'Eswari college': 'Easwari Engineering College',

    # GTEC
    "Ganadhipathy tulsi's jain engineering college":
        "Ganadipathy Tulsi's Jain Engineering College",
    'Ganadipathy Tulsi jain engineering college':
        "Ganadipathy Tulsi's Jain Engineering College",
    "Ganadipathy Tulsi's Jain Engineering college":
        "Ganadipathy Tulsi's Jain Engineering College",

    # Guru Nanak
    'GURU NANAK': 'Guru Nanak College',
    'Guru Nanak College': 'Guru Nanak College',

    # JNN
    'J. N. N college of arts and science':
        'J.N.N. College of Arts and Science',
    'JNN Institute of Engineering':
        'J.N.N. Institute of Engineering',
    'JNNS': 'J.N.N. College of Engineering',

    # Jayaraj Annapackiam
    'Jayaraj Annapackiam college':
        'Jayaraj Annapackiam College for Women',

    # Jeppiaar
    'Jeppiaar Engineering College':
        'Jeppiaar Engineering College',
    'Jeppiaar University':
        'Jeppiaar University',

    # KCG
    'KCG': 'KCG College of Technology',
    'KCG COLLEGE OF TECHNOLORY':
        'KCG College of Technology',

    # Lady Doak
    'Ladydoak college': 'Lady Doak College',

    # Loyola
    'Loyala college': 'Loyola College',

    # MGR
    'MGR': 'M.G.R. Janaki College of Arts and Science',
    'MGR COLLEGE': 'M.G.R. Janaki College of Arts and Science',
    'MGR Janaki College': 'M.G.R. Janaki College of Arts and Science',

    # Meenakshi
    'Meenakshi College': 'Meenakshi College of Engineering',
    'Meenakshi college of engineering':
        'Meenakshi College of Engineering',

    # Mepco
    'Mepco College': 'Mepco Schlenk Engineering College',

    # PSNA
    'PSNA college': 'PSNA College of Engineering and Technology',

    # PSR
    'PSR College': 'PSR Engineering College',

    # Paavai
    'Paavai college': 'Paavai Engineering College',

    # RMD
    'R.M.D. Engineering College': 'R.M.D. Engineering College',
    'RMD College': 'R.M.D. Engineering College',

    # RMK
    'RMK': 'R.M.K. College of Engineering and Technology',
    'RMK Engineering college':
        'R.M.K. College of Engineering and Technology',
    'RMK(cet)': 'R.M.K. College of Engineering and Technology',
    'RMKCET': 'R.M.K. College of Engineering and Technology',
    'Rmk college of engineering and technology':
        'R.M.K. College of Engineering and Technology',

    # Rajalakshmi
    'Rajalakshmi Engineering college':
        'Rajalakshmi Engineering College',

    # Rohini
    'Rohini College': 'Rohini College of Engineering and Technology',

    # SA
    'SA Engineering college': 'S.A. Engineering College',

    # Sathyabama
    'Sathyabama College': 'Sathyabama Institute of Science and Technology',

    # Saveetha
    'saveetha Engineering College':
        'Saveetha Engineering College',

    # Sri Muthukumaran
    'Sri muthukumaran Institute of technology':
        'Sri Muthukumaran Institute of Technology',
    'Sri muthukumaran institute of technology':
        'Sri Muthukumaran Institute of Technology',

    # SRM Madurai
    'Srm madurai college for engineering and technology':
        'SRM Madurai College for Engineering and Technology',

    # St Joseph's
    "St Joseph's College of engineering":
        "St. Joseph's College of Engineering",
    "St. Joseph's college of engineering":
        "St. Joseph's College of Engineering",

    # St Peter's
    "St Peter's College of Engineering and Technology":
        "St. Peter's College of Engineering and Technology",
    'St.peters college':
        "St. Peter's College of Engineering and Technology",

    # TJS
    'T. J. S. Engineering College':
        'T.J.S. Engineering College',
    'T.J.S ENGINEERING COLLEGE':
        'T.J.S. Engineering College',
    'T.J.S. Engineering College':
        'T.J.S. Engineering College',

    # University College Nagercoil
    'University College(NAGERCOIL)':
        'University College of Engineering, Nagercoil',
    'University College(NAGERKOIL)':
        'University College of Engineering, Nagercoil',
    'University college (nagercoil)':
        'University College of Engineering, Nagercoil',

    # Vel Tech
    'Vel Tech Multi tech':
        'Vel Tech Multi Tech Dr. Rangarajan Dr. Sakunthala Engineering College',

    # Velammal
    'Velammal College of Engineering':
        'Velammal College of Engineering and Technology',
    'Velammal college':
        'Velammal College of Engineering and Technology',

    # Vels
    'Vels': 'Vels University Pallavaram',
    'vels university': 'Vels University Pallavaram',

    # Already confirmed
    'Panimalar Engineering College':
        'Panimalar Engineering College',
    'SRM College of Engineering and Technology':
        'SRM College of Engineering and Technology',
    'St. Thomas College of Arts and Science':
        'St. Thomas College of Arts and Science',
}

In [ ]:
final_master['College'] = final_master['College'].replace(college_mapping)
final_master['College'] = final_master['College'].str.strip()

In [ ]:
sorted(final_master['College'].dropna().unique())

In [ ]:
college_mapping = {

    # A.V.C
    'A. V. C college of engineering': 'A.V.C College of Engineering',
    'A.V .C COLLEGE OF ENGINEERING': 'A.V.C College of Engineering',
    'A.V.C College of Engineering': 'A.V.C College of Engineering',

    # AAA
    'AAA Engineering college': 'AAA College of Engineering and Technology',

    # ADHI / ADHIPARASAKTHI
    'Adhi Parasakthi College': 'Adhiparasakthi Engineering College',
    'Adhi College of Engineering and Technology': 'Adhi College of Engineering and Technology',
    'Aadhiparasakthi Eng College': 'Adhiparasakthi Engineering College',
    'Adhiparasakthi College of Engineering': 'Adhiparasakthi Engineering College',
    'Adhiparasakthi college of Engineering': 'Adhiparasakthi Engineering College',
    'Adhiparasakthi college of engineering': 'Adhiparasakthi Engineering College',
    'ADHIPARASAKTHI ENGINEERING COLLEGE, MELMARUVATHUR':
        'Adhiparasakthi Engineering College',

    # AMJ / ANJAC
    'Amj': 'Agurchand Manmull Jain College',
    'ANJAC': 'Ayya Nadar Janaki Ammal College',

    # AMRITA
    'Amrita Vishwa Vidya College': 'Amrita Vishwa Vidyapeetham',

    # ANNA ADARSH
    'Anna Adarsh College': 'Anna Adarsh College for Women',

    # ARULMURUGAN
    'Arulmurugan college of engineering':
        'Arulmigu Kalasalingam College of Engineering',

    # ARUNAI
    'Arunai college': 'Arunai Engineering College',
    'Arunai engineering college': 'Arunai Engineering College',

    # AUXILIUM
    'Auxillium college': 'Auxilium College',

    # AVICHI
    'Avichi college': 'Avichi College of Arts and Science',

    # CTTE
    'CTTE College': 'CTTE College for Women',

    # CHENDURAN
    'Chenduran college of Engineering and Technology':
        'Chenduran College of Engineering and Technology',

    # DHANALAKSHMI COLLEGE OF ENGINEERING
    'Dhanalakshmi College of Engineering':
        'Dhanalakshmi College of Engineering',
    'Dhanalakshmi Engineering college':
        'Dhanalakshmi College of Engineering',
    'Dhanalakshmi college(certificate)':
        'Dhanalakshmi College of Engineering',

    # DHANALAKSHMI SRINIVASAN ENGINEERING COLLEGE
    'Dhanalakshmi Srinivasan Engineering College':
        'Dhanalakshmi Srinivasan Engineering College',
    'Dhanalakshmi Srinivasan Engineering College (A), Perambalur':
        'Dhanalakshmi Srinivasan Engineering College',
    'Dhanalakshmi Srinivasan Engineering College(A), Perambalur':
        'Dhanalakshmi Srinivasan Engineering College',
    'Dhanalakshmi Srinivasan Engineering College, Perambalur':
        'Dhanalakshmi Srinivasan Engineering College',
    'Dhanalakshmi Srinivasan Engineering college':
        'Dhanalakshmi Srinivasan Engineering College',
    'Dhanalakshmi Srinivasan engineering collge (A) Perambalur':
        'Dhanalakshmi Srinivasan Engineering College',
    'Dhanalakshmi srinivasan engineering college (autonomous)':
        'Dhanalakshmi Srinivasan Engineering College',

    # DHANALAKSHMI SRINIVASAN UNIVERSITY
    'Dhanalakshmi Srinivasan University':
        'Dhanalakshmi Srinivasan University',
    'Dhanalakshmi Srinivasan university':
        'Dhanalakshmi Srinivasan University',
    'Dhanalakshmi srinivasan university':
        'Dhanalakshmi Srinivasan University',
    'Dhanalakshmi srinivasan university trichy':
        'Dhanalakshmi Srinivasan University',
    'Dhanalakshmi Srinivasa college':
        'Dhanalakshmi Srinivasan University',

    # DON BOSCO
    'Don bosco college': 'Don Bosco College',

    # EASWARI
    'Eswari college': 'Easwari Engineering College',

    # GTEC
    "Ganadhipathy tulsi's jain engineering college":
        "Ganadipathy Tulsi's Jain Engineering College",
    'Ganadipathy Tulsi jain engineering college':
        "Ganadipathy Tulsi's Jain Engineering College",
    "Ganadipathy Tulsi's Jain Engineering College":
        "Ganadipathy Tulsi's Jain Engineering College",
    "Ganadipathy Tulsi's Jain Engineering college":
        "Ganadipathy Tulsi's Jain Engineering College",

    # GOVERNMENT
    'Government college of engineering bargur Krishnagiri':
        'Government College of Engineering, Bargur',

    # GURU NANAK
    'GURU NANAK': 'Guru Nanak College',
    'Guru Nanak College': 'Guru Nanak College',

    # JNN
    'J. N. N college of arts and science':
        'J.N.N. College of Arts and Science',
    'JNN Institute of Engineering':
        'J.N.N. Institute of Engineering',
    'JNNS':
        'J.N.N. Institute of Engineering',

    # JAYARAJ ANNAPACKIAM
    'Jayaraj Annapackiam college':
        'Jayaraj Annapackiam College for Women',

    # KCG
    'KCG': 'KCG College of Technology',
    'KCG COLLEGE OF TECHNOLORY':
        'KCG College of Technology',

    # KARUNYA
    'Karunya college': 'Karunya Institute of Technology and Sciences',

    # LADY DOAK
    'Ladydoak college': 'Lady Doak College',

    # LOYOLA
    'Loyala college': 'Loyola College',

    # MGR
    'MGR': 'M.G.R. Janaki College of Arts and Science',
    'MGR COLLEGE': 'M.G.R. Janaki College of Arts and Science',
    'MGR Janaki College':
        'M.G.R. Janaki College of Arts and Science',

    # MEENAKSHI
    'Meenakshi College': 'Meenakshi College of Engineering',
    'Meenakshi college of engineering':
        'Meenakshi College of Engineering',

    # MEPCO
    'Mepco College': 'Mepco Schlenk Engineering College',

    # MOTHER TERESA
    'Mother Terasa college of Engineering':
        'Mother Teresa College of Engineering and Technology',

    # PSNA
    'PSNA college': 'PSNA College of Engineering and Technology',

    # PSR
    'PSR College': 'PSR Engineering College',

    # PT LEE
    'PT Lee chengalvaraya':
        'P.T. Lee Chengalvaraya Naicker College of Engineering and Technology',

    # PAAVAI
    'Paavai college': 'Paavai Engineering College',

    # PANIMALAR
    'Panimalar Engineering College':
        'Panimalar Engineering College',
    'Panimalar engineering college':
        'Panimalar Engineering College',
    'Panimalar Engineering collage':
        'Panimalar Engineering College',
    'Panimalar Engineering college':
        'Panimalar Engineering College',
    'PANIMALAR ENGINEERING COLLEGE':
        'Panimalar Engineering College',

    # PARVATHI
    'Parvathi college': 'Parvathi College of Engineering',

    # PONJESLY
    'Ponjesly college': 'Ponjesly College of Engineering',

    # RAJALAKSHMI
    'Rajalakshmi Engineering College':
        'Rajalakshmi Engineering College',
    'Rajalakshmi Engineering college':
        'Rajalakshmi Engineering College',

    # RMD
    'R.M.D. Engineering College':
        'R.M.D. Engineering College',
    'RMD ENGINEERING COLLEGE':
        'R.M.D. Engineering College',
    'RMD College':
        'R.M.D. Engineering College',

    # RMK
    'RMK':
        'R.M.K. College of Engineering and Technology',
    'RMK Engineering college':
        'R.M.K. College of Engineering and Technology',
    'RMK(cet)':
        'R.M.K. College of Engineering and Technology',
    'RMKCET':
        'R.M.K. College of Engineering and Technology',
    'Rmk college of engineering and technology':
        'R.M.K. College of Engineering and Technology',
    'RMK College':
        'R.M.K. College of Engineering and Technology',
    'R.M.K. College of Engineering and Technology':
        'R.M.K. College of Engineering and Technology',

    # ROHINI
    'Rohini College':
        'Rohini College of Engineering and Technology',

    # SA
    'SA Engineering college':
        'S.A. Engineering College',

    # SATHYABAMA
    'Sathyabama College':
        'Sathyabama Institute of Science and Technology',

    # SAVEETHA
    'Saveetha Engineering College':
        'Saveetha Engineering College',
    'saveetha Engineering College':
        'Saveetha Engineering College',

    # SINDHI
    'Sindhi College':
        'Sindhi College of Arts and Science',

    # SOKA IKEDA
    'Soka Ikeda college':
        'Soka Ikeda College',

    # SRI MUTHUKUMARAN
    'Sri Muthukumaran Institute of Technology':
        'Sri Muthukumaran Institute of Technology',
    'Sri muthukumaran Institute of technology':
        'Sri Muthukumaran Institute of Technology',
    'Sri muthukumaran institute of technology':
        'Sri Muthukumaran Institute of Technology',

    # SRM
    'SRM':
        'SRM College of Engineering and Technology',
    'SRM College':
        'SRM College of Engineering and Technology',

    # SRM MADURAI
    'SRM Madurai College for Engineering and Technology':
        'SRM Madurai College for Engineering and Technology',
    'Srm madurai college for engineering and technology':
        'SRM Madurai College for Engineering and Technology',

    # ST JOSEPH
    "St Joseph's College of engineering":
        "St. Joseph's College of Engineering",
    "St. Joseph's college of engineering":
        "St. Joseph's College of Engineering",

    # ST PETER'S
    "St Peter's College of Engineering and Technology":
        "St. Peter's College of Engineering and Technology",
    'St.peters college':
        "St. Peter's College of Engineering and Technology",

    # ST THOMAS
    'St. Thomas College of Arts and Science':
        'St. Thomas College of Arts and Science',
    'St.Thomas College':
        'St. Thomas College of Arts and Science',

    # TJS
    'TJS College':
        'T.J.S. Engineering College',
    'T. J. S. Engineering College':
        'T.J.S. Engineering College',
    'T.J.S ENGINEERING COLLEGE':
        'T.J.S. Engineering College',
    'T.J.S. Engineering College':
        'T.J.S. Engineering College',

    # UNIVERSITY COLLEGE NAGERCOIL
    'University College(NAGERCOIL)':
        'University College of Engineering, Nagercoil',
    'University College(NAGERKOIL)':
        'University College of Engineering, Nagercoil',
    'University college (nagercoil)':
        'University College of Engineering, Nagercoil',

    # VEL TECH
    'Vel Tech Multi tech':
        'Vel Tech Multi Tech Dr. Rangarajan Dr. Sakunthala Engineering College',

    # VELAMMAL
    'Velammal College of Engineering':
        'Velammal College of Engineering and Technology',
    'Velammal college':
        'Velammal College of Engineering and Technology',
    'Velammal(certificate)':
        'Velammal College of Engineering and Technology',

    # VELS
    'Vels':
        'Vels University Pallavaram',
    'Vels University Pallavaram':
        'Vels University Pallavaram',
    'vels university':
        'Vels University Pallavaram',
}

In [ ]:
final_master['College'] = (
    final_master['College']
    .replace(college_mapping)
    .str.strip()
)

In [ ]:
final_master['College'].value_counts(dropna=False).sort_index()

In [ ]:
sorted(final_master['College'].dropna().unique())

In [ ]:
sorted(final_master['College'].dropna().unique())

In [ ]:
abbreviation_mapping = {
    'CT university': 'CT University',
    'CTTE College for Women': 'Chevalier T. Thomas Elizabeth College for Women',
    'SSMIET': 'S S M Institute of Engineering and Technology',
    'TBMLC': 'T.B.M.L. College, Porayar',
    'VIT': 'Vellore Institute of Technology',
    'University VOC College': 'University V.O.C. College of Engineering, Thoothukudi',
    'sivanthi adithnar': 'Sivanthi Aditanar College'
}

final_master['College'] = (
    final_master['College']
    .replace(abbreviation_mapping)
    .str.strip()
)

In [ ]:
sorted(final_master['College'].dropna().unique())

In [ ]:
college_counts = final_master['College'].value_counts(dropna=False).sort_index()
college_counts

In [ ]:
final_master['College'].value_counts(dropna=False).sort_index().to_string()

In [ ]:
# 1. Fix the obvious JCT duplicate
final_master['College'] = final_master['College'].replace({
    'JCT college of engineering and technology  Coimbatore':
        'JCT College of Engineering and Technology',
    'JCT college of engineering and technology Coimbatore':
        'JCT College of Engineering and Technology',
    'Rathinam college of liberal arts and science .':
        'Rathinam College of Liberal Arts and Science'
})

In [ ]:
# 2. Inspect the two Certificate records
final_master[
    final_master['College'].eq('Certificate')
][['S.No', 'Name', 'Phone number', 'College', 'DOJ', 'Student ID']]

In [ ]:
final_master.loc[
    final_master['College'].eq('Certificate'),
    'College'
] = pd.NA

In [ ]:
final_master['College'].value_counts(dropna=False).sort_index()

In [ ]:
final_master['Total billing'].describe()

In [ ]:
final_master['Total billing'].value_counts().sort_index()

In [ ]:
final_master[
    final_master['Total billing'] == 0
][[
    'S.No',
    'Name',
    'College',
    'Total billing',
    'Total collection',
    'Pending amount',
    'Fees Status',
    'Payment_Total'
]]

In [ ]:
final_master['Total collection'].describe()

In [ ]:
final_master['Total collection'].value_counts(dropna=False).sort_index()

In [ ]:
final_master[
    final_master['Total collection'] > final_master['Total billing']
][[
    'S.No',
    'Name',
    'Total billing',
    'Total collection',
    'Pending amount',
    'Fees Status',
    'Payment_Total'
]]

In [ ]:
check_names = ['Parthasarathy', 'Jahnavi', 'deepak K', 'Reshma K']

final_master[
    final_master['Name'].isin(check_names)
][[
    'S.No',
    'Name',
    'Student ID',
    'College',
    'Phone number',
    'DOJ',
    'Total billing',
    'Total collection',
    'Pending amount',
    'Fees Status',
    'Payment_Total'
]].sort_values('Name')

In [ ]:
final_master['Calculated Pending'] = (
    final_master['Total billing'] -
    final_master['Total collection']
)

In [ ]:
final_master[
    final_master['Pending amount'].notna() &
    (final_master['Pending amount'] != final_master['Calculated Pending'])
][[
    'S.No',
    'Name',
    'Total billing',
    'Total collection',
    'Pending amount',
    'Calculated Pending'
]]

In [ ]:
final_master[
    final_master['Pending amount'] < 0
][[
    'S.No',
    'Name',
    'Total billing',
    'Total collection',
    'Pending amount'
]]

In [ ]:
final_master[
    final_master['Name'].isin(
        ['Parthasarathy', 'Jahnavi', 'deepak K', 'Reshma K']
    )
][[
    'S.No',
    'Name',
    'Total billing',
    'Installment 1 GPay',
    'Installment 1 Cash',
    'Installment 2 GPay',
    'Installment 2 Cash',
    'Installment 3 GPay',
    'Installment 3 Cash',
    'Total collection',
    'Pending amount'
]]

In [ ]:
final_master['Overpayment'] = (
    final_master['Total collection'] - final_master['Total billing']
).clip(lower=0)

In [ ]:
final_master['Pending amount'] = (
    final_master['Total billing'] - final_master['Total collection']
).clip(lower=0)

In [ ]:
import pandas as pd

final_master = pd.read_pickle("final_master_cleaned_backup.pkl")

In [ ]:
final_master.shape

In [ ]:
installment_cols = [
    'Installment 1 GPay',
    'Installment 1 Cash',
    'Installment 2 GPay',
    'Installment 2 Cash',
    'Installment 3 GPay',
    'Installment 3 Cash'
]

final_master['Calculated Collection'] = (
    final_master[installment_cols]
    .fillna(0)
    .sum(axis=1)
)

## 11. Validate the final dataset

Final consistency checks, duplicates, missing values, and collection differences.

In [ ]:
collection_mismatch = final_master[
    final_master['Total collection'].notna() &
    (final_master['Calculated Collection'] != final_master['Total collection'])
]

collection_mismatch[
    [
        'S.No',
        'Name',
        'Total billing',
        'Calculated Collection',
        'Total collection',
        'Pending amount'
    ]
]

In [ ]:
final_master.drop(columns='Calculated Collection', inplace=True)

In [ ]:
final_master.columns.tolist()

In [ ]:
final_master.drop(
    columns=['Calculated Pending', 'Overpayment'],
    inplace=True,
    errors='ignore'
)

In [ ]:
final_master.shape

In [ ]:
final_master.columns.tolist()

In [ ]:
final_master.to_pickle("final_master_cleaned_backup.pkl")

In [ ]:
final_master['Pending amount'].isna().sum()

In [ ]:
final_master['Pending amount'].value_counts(dropna=False).sort_index()

In [ ]:
final_master[
    final_master['Pending amount'].isna()
][[
    'S.No',
    'Name',
    'College',
    'Total billing',
    'Total collection',
    'Pending amount',
    'Fees Status',
    'Student ID'
]]

In [ ]:
final_master[
    final_master['Payment_Total'].notna()
][[
    'S.No',
    'Name',
    'Total collection',
    'Payment_Total',
    'Payment_Count'
]].head(20)

In [ ]:
missing_collection_with_payment = final_master[
    final_master['Total collection'].isna() &
    final_master['Payment_Total'].notna()
]

missing_collection_with_payment[
    [
        'S.No',
        'Name',
        'Student ID',
        'Total billing',
        'Total collection',
        'Payment_Total',
        'Payment_Count'
    ]
]

In [ ]:
missing_collection_with_payment.shape

In [ ]:
mask = (
    final_master['Total collection'].isna() &
    final_master['Payment_Total'].notna() &
    (final_master['Payment_Total'] == final_master['Total billing'])
)

mask.sum()

In [ ]:
final_master.loc[mask, 'Total collection'] = (
    final_master.loc[mask, 'Payment_Total']
)

In [ ]:
pending_mask = (
    final_master['Pending amount'].isna() &
    final_master['Total collection'].notna() &
    final_master['Total billing'].notna()
)

final_master.loc[pending_mask, 'Pending amount'] = (
    final_master.loc[pending_mask, 'Total billing'] -
    final_master.loc[pending_mask, 'Total collection']
).clip(lower=0)

In [ ]:
final_master[
    final_master['Student ID'].isin(
        [f'IN{i:04d}' for i in range(543, 573)]
    )
][[
    'Student ID',
    'Name',
    'Total billing',
    'Total collection',
    'Pending amount',
    'Payment_Total',
    'Payment_Count'
]]

In [ ]:
installment_cols = [
    'Installment 1 GPay',
    'Installment 1 Cash',
    'Installment 2 GPay',
    'Installment 2 Cash',
    'Installment 3 GPay',
    'Installment 3 Cash'
]

final_master['Installment Total'] = (
    final_master[installment_cols]
    .fillna(0)
    .sum(axis=1)
)

In [ ]:
final_master['Collection Difference'] = (
    final_master['Total collection'] -
    final_master['Installment Total']
)

In [ ]:
untallied = final_master[
    final_master['Total collection'].notna() &
    (final_master['Collection Difference'] != 0)
]

untallied[[
    'S.No',
    'Name',
    'Total billing',
    'Total collection',
    'Installment Total',
    'Collection Difference'
]]

In [ ]:
installment_cols = [
    'Installment 1 GPay',
    'Installment 1 Cash',
    'Installment 2 GPay',
    'Installment 2 Cash',
    'Installment 3 GPay',
    'Installment 3 Cash'
]

# Check whether at least one installment value exists
has_installment = final_master[installment_cols].notna().any(axis=1)

# Reconciliation difference
final_master['Collection Difference'] = pd.NA

final_master.loc[has_installment, 'Collection Difference'] = (
    final_master.loc[has_installment, 'Total collection']
    - final_master.loc[has_installment, 'Installment Total']
)

In [ ]:
final_master[
    final_master['Collection Difference'].notna() &
    (final_master['Collection Difference'] != 0)
][[
    'S.No',
    'Name',
    'Total billing',
    'Total collection',
    'Installment Total',
    'Collection Difference'
]]

## 12. Final checks and export

Final row/column checks and output preparation.

In [ ]:
final_master.to_pickle("final_master_cleaned_backup.pkl")

In [ ]:
final_master.to_pickle("final_master_cleaned_backup.pkl")

In [ ]:
final_master.shape

In [ ]:
final_master[final_master['Student ID'].notna()]['Student ID'].duplicated().sum()

In [ ]:
final_master[
    final_master['Student ID'].notna() &
    final_master['Student ID'].duplicated(keep=False)
][[
    'S.No',
    'Student ID',
    'Name',
    'Phone number',
    'College',
    'Total billing',
    'Total collection'
]].sort_values('Student ID')

In [ ]:
print("Missing:", final_master['Phone number'].isna().sum())

print(
    "Unique phone numbers:",
    final_master['Phone number'].nunique(dropna=True)
)

print(
    "Duplicate phone numbers:",
    final_master['Phone number'].duplicated(keep=False).sum()
)

In [ ]:
final_master[
    final_master['Phone number'].notna() &
    final_master['Phone number'].duplicated(keep=False)
][[
    'S.No',
    'Name',
    'Phone number',
    'College',
    'Student ID'
]].sort_values('Phone number')

In [ ]:
phone_dupes = final_master[
    final_master['Phone number'].notna() &
    final_master['Phone number'].duplicated(keep=False)
].copy()

phone_dupes = phone_dupes.sort_values(
    ['Phone number', 'College', 'DOMAIN', 'Name']
)

phone_dupes[[
    'S.No',
    'Name',
    'Phone number',
    'College',
    'DOMAIN',
    'Student ID'
]]

In [ ]:
# Privacy note: the original cell inspected specific student phone numbers.
# That identifying lookup was omitted from this public copy.
# Perform any authorised record-level investigation locally, without publishing identifiers.

In [ ]:
duplicate_snos = [49, 258, 462, 259]

final_master = final_master[
    ~final_master['S.No'].isin(duplicate_snos)
].copy()

In [ ]:
final_master.shape

In [ ]:
final_master.to_pickle("final_master_cleaned_backup.pkl")

In [ ]:
# Privacy note: the original cell inspected specific student phone numbers.
# That identifying lookup was omitted from this public copy.
# Perform any authorised record-level investigation locally, without publishing identifiers.

In [ ]:
duplicate_snos = [50, 463, 260]

final_master = final_master[
    ~final_master['S.No'].isin(duplicate_snos)
].copy()

In [ ]:
final_master.shape

In [ ]:
# Privacy note: the original cell inspected specific student phone numbers.
# That identifying lookup was omitted from this public copy.
# Perform any authorised record-level investigation locally, without publishing identifiers.

In [ ]:
final_master.to_pickle("final_master_cleaned_backup.pkl")

In [ ]:
duplicate_snos = [50, 463, 260]

final_master = final_master[
    ~final_master['S.No'].isin(duplicate_snos)
].copy()

In [ ]:
print(final_master.shape)

In [ ]:
final_master.drop(
    columns=['Installment Total'],
    inplace=True,
    errors='ignore'
)

In [ ]:
print("Duplicate Student IDs:",
      final_master['Student ID'].duplicated().sum())

print("Duplicate complete rows:",
      final_master.duplicated().sum())

In [ ]:
final_master.to_pickle("final_master_cleaned_backup.pkl")
final_master.to_excel("final_master_cleaned.xlsx", index=False)

print("FINAL SHAPE:", final_master.shape)

In [ ]:
final_master.info()

In [ ]:
final_master = final_master.reset_index(drop=True)

final_master['S.No'] = range(1, len(final_master) + 1)

In [ ]:
print(final_master['S.No'].head())
print(final_master['S.No'].tail())
print(final_master.shape)

In [ ]:
final_master.to_pickle("final_master_cleaned_backup.pkl")
final_master.to_excel("final_master_cleaned.xlsx", index=False)

In [ ]:
final_master.to_pickle("final_master_cleaned_backup.pkl")
final_master.to_excel("final_master_cleaned.xlsx", index=False)

print("Rows:", len(final_master))
print("Columns:", len(final_master.columns))
print("S.No range:", final_master['S.No'].min(), "to", final_master['S.No'].max())

In [ ]:
# Privacy note: the original cell inspected or corrected named individual records.
# This record-specific operation was omitted from the public copy.
# Apply approved corrections locally using a documented, non-public exception list.

In [ ]:
# Privacy note: the original cell inspected or corrected named individual records.
# This record-specific operation was omitted from the public copy.
# Apply approved corrections locally using a documented, non-public exception list.

In [ ]:
# Privacy note: the original cell inspected or corrected named individual records.
# This record-specific operation was omitted from the public copy.
# Apply approved corrections locally using a documented, non-public exception list.

In [ ]:
final_master.loc[mask, 'Collection Difference'] = 0

In [ ]:
# Privacy note: the original cell inspected or corrected named individual records.
# This record-specific operation was omitted from the public copy.
# Apply approved corrections locally using a documented, non-public exception list.

In [ ]:
# Privacy note: the original cell inspected or corrected named individual records.
# This record-specific operation was omitted from the public copy.
# Apply approved corrections locally using a documented, non-public exception list.

In [ ]:
import pandas as pd

installment_cols = [
    'Installment 1 GPay',
    'Installment 1 Cash',
    'Installment 2 GPay',
    'Installment 2 Cash',
    'Installment 3 GPay',
    'Installment 3 Cash'
]

# 1. Check whether the student has any installment payment data
has_installment = final_master[installment_cols].notna().any(axis=1)

# 2. Calculate total from Installments 1, 2 and 3
installment_total = (
    final_master[installment_cols]
    .fillna(0)
    .sum(axis=1)
)

# 3. Fill ONLY missing Total collection values
#    where installment information is available
fill_collection = (
    final_master['Total collection'].isna() &
    has_installment
)

final_master.loc[fill_collection, 'Total collection'] = (
    installment_total[fill_collection]
)

# 4. Calculate Pending amount where billing + collection are available
can_calculate_pending = (
    final_master['Total billing'].notna() &
    final_master['Total collection'].notna()
)

final_master.loc[can_calculate_pending, 'Pending amount'] = (
    final_master.loc[can_calculate_pending, 'Total billing']
    - final_master.loc[can_calculate_pending, 'Total collection']
).clip(lower=0)

# 5. Recalculate Collection Difference
#    only where installment data actually exists
final_master.loc[has_installment, 'Collection Difference'] = (
    final_master.loc[has_installment, 'Total collection']
    - installment_total[has_installment]
)

# 6. Show how many records were updated
print("Collections filled:", fill_collection.sum())
print("Pending amounts available:", can_calculate_pending.sum())
print("Installment records:", has_installment.sum())

In [ ]:
final_master[
    fill_collection
][[
    'S.No',
    'Name',
    'Total billing',
    'Total collection',
    'Pending amount',
    'Installment 1 GPay',
    'Installment 1 Cash',
    'Installment 2 GPay',
    'Installment 2 Cash',
    'Installment 3 GPay',
    'Installment 3 Cash',
    'Collection Difference'
]]

In [ ]:
final_master.to_pickle("final_master_cleaned_backup.pkl")
final_master.to_excel("final_master_cleaned.xlsx", index=False)

print("Rows:", len(final_master))
print("Columns:", len(final_master.columns))
print("S.No:", final_master['S.No'].min(), "to", final_master['S.No'].max())

## 13. Additional date checks and final output

Follow-up date validation and output operations from the original working notebook.

In [ ]:
dataset = final_master

In [ ]:
final_master

In [ ]:
import os

print(os.path.abspath("final_master_cleaned_backup.pkl"))

In [ ]:
import os

desktop = os.path.join(os.path.expanduser("~"), "Desktop")

print("Desktop path:")
print(desktop)

print("\nDesktop exists:", os.path.exists(desktop))

In [ ]:
final_master.to_excel(
    str(PROCESSED_DIR / "final_master_cleaned.xlsx"),
    index=False
)

print("Saved successfully.")

In [ ]:
import pandas as pd

dataset = pd.read_pickle(
    str(PROCESSED_DIR / "final_master_cleaned_backup.pkl")
)

In [ ]:
date_cols = [
    'DOJ',
    'Installment 1 Date',
    'Installment 2 Date',
    'Installment 3 Date',
    'Preferred Start Date'
]

final_master[date_cols].dtypes

In [ ]:
import pandas as pd

final_master = pd.read_pickle(
    str(PROCESSED_DIR / "final_master_cleaned_backup.pkl")
)

print(final_master.shape)

In [ ]:
date_cols = [
    'DOJ',
    'Installment 1 Date',
    'Installment 2 Date',
    'Installment 3 Date',
    'Preferred Start Date'
]

final_master[date_cols].dtypes

In [ ]:
date_cols = [
    'DOJ',
    'Installment 1 Date',
    'Installment 2 Date',
    'Installment 3 Date',
    'Preferred Start Date'
]

for col in date_cols:
    print("\n" + "="*60)
    print(col)
    print("="*60)
    print(final_master[col].to_string(index=False))

In [ ]:
final_master[date_cols]

In [ ]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

final_master[date_cols]

In [ ]:
date_cols = [
    'DOJ',
    'Installment 1 Date',
    'Installment 2 Date',
    'Installment 3 Date',
    'Preferred Start Date'
]

for col in date_cols:
    final_master[col] = pd.to_datetime(
        final_master[col],
        errors='coerce'
    ).dt.date

In [ ]:
final_master[date_cols].head(10)

In [ ]:
installment_cols = [
    'Installment 1 Date',
    'Installment 2 Date',
    'Installment 3 Date'
]

for col in installment_cols:
    mask = final_master[col].astype('string').str.contains(',', na=False)

    if mask.any():
        print("\n" + "="*70)
        print(col)
        print("="*70)
        print(final_master.loc[mask, ['S.No', 'Name', col]].to_string(index=False))

In [ ]:
installment_cols = [
    'Installment 1 Date',
    'Installment 2 Date',
    'Installment 3 Date'
]

for col in installment_cols:
    mask = final_master[col].astype('string').str.contains(',', na=False)

    if mask.any():
        print("\n" + "="*70)
        print(col)
        print("="*70)
        print(final_master.loc[mask, ['S.No', 'Name', col]].to_string(index=False))

In [ ]:
import pandas as pd

original = pd.read_pickle(
    str(PROCESSED_DIR / "final_master_cleaned_backup.pkl")
)

installment_cols = [
    'Installment 1 Date',
    'Installment 2 Date',
    'Installment 3 Date'
]

for col in installment_cols:
    mask = original[col].astype('string').str.contains(
        r',|;| and ',
        case=False,
        na=False
    )

    if mask.any():
        print("\n" + "="*70)
        print(col)
        print("="*70)
        print(
            original.loc[
                mask,
                ['S.No', 'Name', col]
            ].to_string(index=False)
        )

In [ ]:
for col in installment_cols:
    print("\n", "="*70)
    print(col)
    print("="*70)

    print(
        original.loc[
            original[col].notna(),
            ['S.No', 'Name', col]
        ].to_string(index=False)
    )

In [ ]:
import pandas as pd

final_master = pd.read_pickle(
    str(PROCESSED_DIR / "final_master_cleaned_backup.pkl")
)

In [ ]:
installment_cols = [
    'Installment 1 Date',
    'Installment 2 Date',
    'Installment 3 Date'
]

for col in installment_cols:
    final_master[col] = (
        final_master[col]
        .astype('string')
        .str.split(',')
        .str[0]
        .str.strip()
    )

    final_master[col] = pd.to_datetime(
        final_master[col],
        errors='coerce',
        dayfirst=True
    ).dt.date

In [ ]:
# Privacy note: the original cell inspected or corrected named individual records.
# This record-specific operation was omitted from the public copy.
# Apply approved corrections locally using a documented, non-public exception list.

In [ ]:
# Reload the untouched backup
final_master = pd.read_pickle(
    str(PROCESSED_DIR / "final_master_cleaned_backup.pkl")
)

installment_cols = [
    'Installment 1 Date',
    'Installment 2 Date',
    'Installment 3 Date'
]

# Keep only the FIRST date if multiple dates exist
for col in installment_cols:
    final_master[col] = (
        final_master[col]
        .astype('string')
        .str.split(',')
        .str[0]
        .str.strip()
    )

    final_master[col] = pd.to_datetime(
        final_master[col],
        errors='coerce',
        dayfirst=True
    ).dt.date

In [ ]:
# Privacy note: the original cell inspected or corrected named individual records.
# This record-specific operation was omitted from the public copy.
# Apply approved corrections locally using a documented, non-public exception list.

In [ ]:
date_cols = [
    'DOJ',
    'Installment 1 Date',
    'Installment 2 Date',
    'Installment 3 Date',
    'Preferred Start Date'
]

print(final_master[date_cols].dtypes)

In [ ]:
date_cols = [
    'DOJ',
    'Installment 1 Date',
    'Installment 2 Date',
    'Installment 3 Date',
    'Preferred Start Date'
]

for col in date_cols:
    final_master[col] = pd.to_datetime(
        final_master[col],
        errors='coerce'
    )

In [ ]:
print(final_master[date_cols].dtypes)

In [ ]:
final_master[date_cols].head(10)

In [ ]:
final_master.to_pickle(
    str(PROCESSED_DIR / "final_master_cleaned_backup.pkl")
)

final_master.to_excel(
    str(PROCESSED_DIR / "final_master_cleaned.xlsx"),
    index=False
)

print("Saved:", final_master.shape)

In [ ]:
final_master.info()

## Reproducibility and privacy notes

- This public notebook is a cleaned presentation of the original working notebook, not a verified end-to-end pipeline. The original notebook contains iterative cells and manual investigation; some cells may depend on variables created earlier or require review before a fresh run.
- Notebook outputs were cleared to avoid publishing student records.
- Individual-specific lookups and corrections were omitted from this public copy.
- Verify merge keys, duplicate handling, date parsing, and financial reconciliation rules against the authorised source workbook before using the exported data.
- Keep raw and processed student data local and private.
